# Yambda-50M: сколько выученных признаков нужно

На 3 сидах сравниваются `grammar32` и `grammar64` (32 и 64 выученных признака по истории) и гибриды с ручными признаками `human+grammar32`, `human+grammar64`.
Значимость - парный перестановочный тест по пользователям против `human` и между K, с поправкой Холма.

Постановка A: объект - прослушивание, которое включил рекомендатель (`is_organic = 0`), метка - дослушал ли пользователь трек (`played_ratio_pct >= 50`).
Признаки считаются на момент конца предыдущего прослушивания пользователя, только по строго предыдущим событиям.
Разбиение: train 28 дней, val 7 дней, test 7 дней; финальные модели переобучаются на train + val, test оценивается один раз.

Ноутбук сгенерирован `kaggle/build_notebook.py` из модуля `rank/` репозитория, код записывается ячейками `%%writefile`.
Артефакты сохраняются в `/kaggle/working/outputs`.
Если на входе есть сохранённые признаки ноутбука `yambda-features` с тем же хешем кода, шаги 1-3 берут их оттуда.

In [ ]:
!pip install -q polars==1.38.1

In [ ]:
import os

os.environ["_RJEM_MALLOC_CONF"] = "background_thread:true,dirty_decay_ms:500,muzzy_decay_ms:500"
os.environ["POLARS_MAX_THREADS"] = "4"

import json
import shutil
import subprocess
import sys
import time
from pathlib import Path

CODE = Path("/kaggle/working/code")
OUT = Path("/kaggle/working/outputs")
WORK = Path("/tmp/work")
for p in [CODE / "rank", OUT, WORK]:
    p.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(CODE))

events_path = next(Path("/kaggle/input").rglob("multi_event.parquet"))
mappings_dir = next(Path("/kaggle/input").rglob("artist_item_mapping.parquet")).parent
print(f"events: {events_path}")
GPU_INFO = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip() if shutil.which("nvidia-smi") else "none"
print(GPU_INFO)

In [ ]:
%%writefile /kaggle/working/code/rank/__init__.py
import os

os.environ.setdefault("_RJEM_MALLOC_CONF", "background_thread:true,dirty_decay_ms:500,muzzy_decay_ms:500")
os.environ.setdefault("POLARS_MAX_THREADS", "4")

In [ ]:
%%writefile /kaggle/working/code/rank/config.py
HOUR = 3600
DAY = 86400
LISTEN_PLUS = 50
SESSION_IDLE = 30 * 60

TEST_DAYS = 7
VAL_DAYS = 7
TRAIN_DAYS = 28
OBJECT_DAYS = TEST_DAYS + VAL_DAYS + TRAIN_DAYS

SMOOTH_M = 20.0
SEED = 42

SPLITS = ("train", "val", "test")

In [ ]:
%%writefile /kaggle/working/code/rank/prepare.py
import gc
import shutil
from pathlib import Path

import polars as pl

from . import config


def first_mapping(path, key, items):
    return (
        pl.scan_parquet(path)
        .join(items.lazy(), on="item_id", how="semi")
        .group_by("item_id")
        .agg(pl.col(key).min())
        .collect()
    )


def enrich(chunk, artist_of, album_of):
    return (
        chunk.join(artist_of, on="item_id", how="left")
        .join(album_of, on="item_id", how="left")
        .sort("uid", "timestamp", maintain_order=True)
        .with_columns(
            (pl.col("played_ratio_pct") >= config.LISTEN_PLUS).alias("listen_plus"),
            pl.int_range(pl.len(), dtype=pl.UInt32).over("uid", "item_id").alias("track_seen_n"),
            pl.when(pl.col("artist_id").is_not_null())
            .then(pl.int_range(pl.len(), dtype=pl.UInt32).over("uid", "artist_id"))
            .alias("artist_seen_n"),
            pl.col("timestamp").cast(pl.Int64).diff().over("uid", "item_id").cast(pl.UInt32).alias("dt_track"),
            pl.when(pl.col("artist_id").is_not_null())
            .then(pl.col("timestamp").cast(pl.Int64).diff().over("uid", "artist_id"))
            .cast(pl.UInt32)
            .alias("dt_artist"),
            (pl.col("timestamp").cast(pl.Int64).diff().over("uid").fill_null(10**9) > config.SESSION_IDLE)
            .cast(pl.UInt32)
            .cum_sum()
            .over("uid")
            .alias("session_idx"),
        )
        .with_columns(
            pl.int_range(pl.len(), dtype=pl.UInt32).over("uid", "session_idx").alias("pos_in_session"),
            pl.col("listen_plus").shift(1).over("uid", "session_idx").alias("prev_listen_plus"),
        )
    )


def scan_listens(events_path, max_ts=None):
    lf = pl.scan_parquet(events_path).filter(pl.col("event_type") == "listen")
    if max_ts is not None:
        lf = lf.filter(pl.col("timestamp") < max_ts)
    return lf.select("uid", "timestamp", "item_id", "is_organic", "played_ratio_pct", "track_length_seconds")


def load_mappings(events_path, mappings_dir):
    items = scan_listens(events_path).select("item_id").unique().collect()
    mappings_dir = Path(mappings_dir)
    artist_of = first_mapping(mappings_dir / "artist_item_mapping.parquet", "artist_id", items)
    album_of = first_mapping(mappings_dir / "album_item_mapping.parquet", "album_id", items)
    return artist_of, album_of


def prepare_listens(events_path, mappings_dir, out_dir, users_per_chunk=1000):
    out_dir = Path(out_dir)
    if out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True)

    listens = scan_listens(events_path)
    artist_of, album_of = load_mappings(events_path, mappings_dir)
    uids = listens.select("uid").unique().sort("uid").collect()["uid"].to_list()
    print(f"users: {len(uids)}, artist map: {artist_of.height}, album map: {album_of.height}")

    for i in range(0, len(uids), users_per_chunk):
        lo, hi = uids[i], uids[min(i + users_per_chunk, len(uids)) - 1]
        chunk = listens.filter(pl.col("uid").is_between(lo, hi)).collect()
        part = enrich(chunk, artist_of, album_of)
        path = out_dir / f"part_{i // users_per_chunk:03d}.parquet"
        part.write_parquet(path, compression="zstd")
        print(f"{path.name}: users {lo}-{hi}, rows {part.height}")
        del chunk, part
        gc.collect()

    print(f"done -> {out_dir}")

In [ ]:
%%writefile /kaggle/working/code/rank/windows.py
import numpy as np

ALL = 1 << 40
TS_MASK = 0xFFFFFFFF
NEGLIGIBLE = 700


class GroupIndex:
    def __init__(self, group, ts, row=None):
        group = np.asarray(group, dtype=np.int64)
        if row is None:
            key = (group << 32) | np.asarray(ts, dtype=np.int64)
            order = np.argsort(key, kind="stable")
            self.key_ts = key[order]
            del key
            self.key_row = None
        else:
            ts = np.asarray(ts, dtype=np.int64)
            row = np.asarray(row, dtype=np.int64)
            order = np.lexsort((row, group))
            base = group[order] << 32
            self.key_ts = base | ts[order]
            self.key_row = base | row[order]
        self.order = order.astype(np.int32) if len(order) < 2**31 else order
        del order
        self.n = len(group)

    def cumsum(self, values):
        out = np.zeros(self.n + 1, dtype=np.float64)
        np.cumsum(np.asarray(values, dtype=np.float64)[self.order], out=out[1:])
        return out

    def sorted_values(self, values):
        return np.asarray(values)[self.order]

    def decay_prefixes(self, values, tau):
        return decay_prefixes(self, values, tau)

    def span(self, q_group, end_row=None, end_ts=None, inclusive=False):
        base = np.asarray(q_group, dtype=np.int64) << 32
        start = np.searchsorted(self.key_ts, base, side="left")
        if end_row is not None:
            hi = np.searchsorted(self.key_row, base | np.asarray(end_row, dtype=np.int64), side="left")
        else:
            hi = np.searchsorted(self.key_ts, base | np.asarray(end_ts, dtype=np.int64), side="right" if inclusive else "left")
        return start, np.maximum(hi, start)

    def window_start(self, q_group, start, hi, from_ts):
        base = np.asarray(q_group, dtype=np.int64) << 32
        lo = np.searchsorted(self.key_ts, base | np.maximum(np.asarray(from_ts, dtype=np.int64), 0), side="left")
        return np.clip(lo, start, hi)


class WindowQuery:
    def __init__(self, index, q_group, ref_ts, end_row=None, end_ts=None, inclusive=False, valid=None):
        self.index = index
        self.q_group = np.where(valid, q_group, 0) if valid is not None else np.asarray(q_group)
        self.valid = valid
        self.ref_ts = np.asarray(ref_ts, dtype=np.int64)
        self.end_row = None if end_row is None else np.asarray(end_row, dtype=np.int64)
        self.start, self.hi = index.span(self.q_group, end_row=end_row, end_ts=end_ts, inclusive=inclusive)
        if valid is not None:
            self.hi = np.where(valid, self.hi, self.start)
        self._lo = {}

    def lo(self, window):
        if window >= ALL:
            return self.start
        if window not in self._lo:
            self._lo[window] = self.index.window_start(self.q_group, self.start, self.hi, self.ref_ts - window)
        return self._lo[window]

    def row_sum(self, csum, k):
        return self.since_row(csum, np.maximum(self.end_row - k, 0))

    def since_row(self, csum, from_row):
        base = np.asarray(self.q_group, dtype=np.int64) << 32
        lo = np.searchsorted(self.index.key_row, base | np.asarray(from_row, dtype=np.int64), side="left")
        lo = np.clip(lo, self.start, self.hi)
        return (csum[self.hi] - csum[lo]).astype(np.float32)

    def sum(self, csum, window):
        return (csum[self.hi] - csum[self.lo(window)]).astype(np.float32)

    def last(self, sorted_values, fill=np.nan):
        has = self.hi > self.start
        out = np.full(len(self.hi), fill, dtype=np.float64)
        out[has] = sorted_values[self.hi[has] - 1]
        return out

    def last_ts(self):
        has = self.hi > self.start
        out = np.full(len(self.hi), np.nan, dtype=np.float64)
        out[has] = self.index.key_ts[self.hi[has] - 1] & TS_MASK
        return out

    def first_ts(self):
        has = self.hi > self.start
        out = np.full(len(self.hi), np.nan, dtype=np.float64)
        out[has] = self.index.key_ts[self.start[has]] & TS_MASK
        return out

    def decayed(self, prefix):
        has = self.hi > self.start
        last = np.maximum(self.hi - 1, 0)
        log_sum = np.where(has, prefix.log_prefix[last] - prefix.offset[last] - self.ref_ts / prefix.tau, -np.inf)
        return self.masked(np.exp(log_sum)).astype(np.float32)

    def masked(self, values):
        if self.valid is None:
            return values
        return np.where(self.valid, values, np.nan)


class DecayPrefix:
    def __init__(self, log_prefix, offset, tau):
        self.log_prefix = log_prefix
        self.offset = offset
        self.tau = float(tau)


def decay_prefixes(index, values, tau):
    ts = (index.key_ts & TS_MASK).astype(np.float64)
    group_gap = ts.max(initial=0) / tau + NEGLIGIBLE
    group = index.key_ts >> 32
    offset = np.r_[0, np.cumsum(group[1:] != group[:-1])].astype(np.float64) * group_gap
    base = ts / tau + offset
    out = {}
    for name, v in values.items():
        with np.errstate(divide="ignore"):
            x = np.log(index.sorted_values(np.asarray(v, dtype=np.float64))) + base
        out[name] = DecayPrefix(np.logaddexp.accumulate(x), offset, tau)
    return out

In [ ]:
%%writefile /kaggle/working/code/rank/cf.py
import numpy as np
import polars as pl
from scipy.sparse import csr_matrix
from sklearn.utils.extmath import randomized_svd

from . import config

RECENT = 10
BATCH = 20000


class ItemFactors:
    def __init__(self, item_ids, item_vecs, user_ids, user_vecs, cutoff_ts):
        self.item_ids = item_ids
        self.item_vecs = item_vecs
        self.item_unit = item_vecs / np.maximum(np.linalg.norm(item_vecs, axis=1, keepdims=True), 1e-9)
        self.user_ids = user_ids
        self.user_vecs = user_vecs
        self.cutoff_ts = cutoff_ts

    @staticmethod
    def lookup(ids, keys):
        keys = np.asarray(keys, dtype=np.int64)
        pos = np.clip(np.searchsorted(ids, keys), 0, len(ids) - 1)
        return np.where(ids[pos] == keys, pos, -1)

    def item_rows(self, items):
        return self.lookup(self.item_ids, items)

    def user_rows(self, users):
        return self.lookup(self.user_ids, users)


def fit_item_factors(listens, cutoff_ts, dim=64, min_users=3):
    pairs = (
        listens.filter(pl.col("timestamp") < cutoff_ts, pl.col("listen_plus"))
        .group_by("uid", "item_id")
        .agg(pl.len().alias("n"))
        .collect(engine="streaming")
    )
    items = pairs.group_by("item_id").agg(pl.len().alias("users")).filter(pl.col("users") >= min_users)
    pairs = pairs.join(items.select("item_id"), on="item_id", how="semi")
    user_ids = np.sort(pairs["uid"].unique().to_numpy()).astype(np.int64)
    item_ids = np.sort(pairs["item_id"].unique().to_numpy()).astype(np.int64)
    rows = np.searchsorted(user_ids, pairs["uid"].to_numpy())
    cols = np.searchsorted(item_ids, pairs["item_id"].to_numpy())
    matrix = csr_matrix((np.log1p(pairs["n"].to_numpy()).astype(np.float32), (rows, cols)), shape=(len(user_ids), len(item_ids)))
    u, s, vt = randomized_svd(matrix, n_components=dim, n_iter=5, random_state=config.SEED)
    print(f"item factors: {len(user_ids)} users, {len(item_ids)} items, nnz {matrix.nnz}, dim {dim}")
    return ItemFactors(item_ids, (vt.T * s).astype(np.float32), user_ids, u.astype(np.float32), cutoff_ts)


def cf_features(cf, uid, item, lp, objects, user_start):
    n_obj = len(objects)
    item_row = cf.item_rows(item)
    target = item_row[objects]
    user_row = cf.user_rows(uid[objects])
    known = target >= 0
    out = {
        "cf_known": known.astype(np.float32),
        "cf_user_score": np.full(n_obj, np.nan, dtype=np.float32),
        "cf_sim_prev": np.full(n_obj, np.nan, dtype=np.float32),
        "cf_sim_mean10": np.full(n_obj, np.nan, dtype=np.float32),
        "cf_sim_lp10": np.full(n_obj, np.nan, dtype=np.float32),
        "cf_sim_skip10": np.full(n_obj, np.nan, dtype=np.float32),
    }
    has_user = known & (user_row >= 0)
    out["cf_user_score"][has_user] = np.einsum("ij,ij->i", cf.user_vecs[user_row[has_user]], cf.item_vecs[target[has_user]])

    offsets = np.arange(1, RECENT + 1)
    for b in range(0, n_obj, BATCH):
        sl = slice(b, b + BATCH)
        o, t = objects[sl], target[sl]
        hist = o[:, None] - offsets[None, :]
        in_user = hist >= user_start[o - 1][:, None]
        hist = np.where(in_user, hist, 0)
        h_row = np.where(in_user, item_row[hist], -1)
        valid = (h_row >= 0) & (t >= 0)[:, None]
        sims = np.einsum("bkd,bd->bk", cf.item_unit[np.maximum(h_row, 0)], cf.item_unit[np.maximum(t, 0)])
        sims = np.where(valid, sims, 0.0)
        w_lp = np.where(valid, lp[hist], 0.0)
        w_skip = np.where(valid, 1.0 - lp[hist], 0.0)
        out["cf_sim_prev"][sl] = np.where(valid[:, 0], sims[:, 0], np.nan)
        out["cf_sim_mean10"][sl] = safe_ratio(sims.sum(1), valid.sum(1))
        out["cf_sim_lp10"][sl] = safe_ratio((sims * w_lp).sum(1), w_lp.sum(1))
        out["cf_sim_skip10"][sl] = safe_ratio((sims * w_skip).sum(1), w_skip.sum(1))
    return out


def safe_ratio(num, den):
    return np.where(den > 0, num / np.maximum(den, 1e-12), np.nan)

In [ ]:
%%writefile /kaggle/working/code/rank/basis.py
import numpy as np

from . import config

H = config.HOUR
D = config.DAY
RELATIONS = ["u", "ui", "ua", "ual", "cf"]
ATOMS = ["rec_lp", "rec_skip", "org_lp", "org_skip", "ratio", "novel_lp", "novel_skip"]
ROW_KERNELS = {"r1": 1, "r3": 3, "r10": 10}
TIME_KERNELS = {"t1h": H, "t6h": 6 * H, "t1d": D, "t3d": 3 * D, "t7d": 7 * D, "t30d": 30 * D, "t90d": 90 * D}
RECENCY_KERNELS = {"x1d": D, "x30d": 30 * D}
KERNELS = list(ROW_KERNELS) + ["sess"] + list(TIME_KERNELS) + ["all"] + list(RECENCY_KERNELS)
CF_HISTORY = 64
CF_BATCH = 5000


class Alphabet:
    def __init__(self, name, atoms, kernels):
        self.name = name
        self.atoms = list(atoms)
        self.kernels = list(kernels)

    @property
    def shape(self):
        return len(RELATIONS), len(self.atoms), len(self.kernels)

    def columns(self):
        return [basis_name(r, a, k) for r in RELATIONS for a in self.atoms for k in self.kernels]


ALPHABETS = {
    "base": Alphabet("base", ATOMS[:4], list(ROW_KERNELS) + list(TIME_KERNELS) + ["all"]),
    "rich": Alphabet("rich", ATOMS, KERNELS),
}


def basis_name(relation, atom, kernel):
    return f"b_{relation}_{atom}_{kernel}"


def basis_columns():
    return ALPHABETS["rich"].columns()


def atom_values(h):
    skip = 1.0 - h.lp
    return {
        "rec_lp": h.rec * h.lp,
        "rec_skip": h.rec * skip,
        "org_lp": h.org * h.lp,
        "org_skip": h.org * skip,
        "ratio": np.clip(h.ratio, 0, 100) / 100,
        "novel_lp": h.new_artist * h.lp,
        "novel_skip": h.new_artist * skip,
    }


def relation_basis(name, relation, atoms, session_from):
    out = {}
    for kernel, k in ROW_KERNELS.items():
        for atom, values in atoms.items():
            out[basis_name(name, atom, kernel)] = relation.row_sum(values, k)
    for atom, values in atoms.items():
        out[basis_name(name, atom, "sess")] = relation.since_row(values, session_from)
    for kernel, tau in TIME_KERNELS.items():
        sums = relation.decayed_sums(atoms, tau)
        for atom, values in sums.items():
            out[basis_name(name, atom, kernel)] = values
    for atom, values in atoms.items():
        out[basis_name(name, atom, "all")] = relation.window_sum(values, "all")
    for kernel, tau in RECENCY_KERNELS.items():
        for atom, values in atoms.items():
            out[basis_name(name, atom, kernel)] = relation.last_decayed(values, tau)
    return {k: np.nan_to_num(v, nan=0.0).astype(np.float32) for k, v in out.items()}


def cf_basis(cf, h, atoms):
    o, ref = h.objects, h.ref
    n_obj = len(o)
    item_row = cf.item_rows(h.item)
    target = item_row[o]
    atom_matrix = np.stack([atoms[a] for a in ATOMS], axis=1)
    out = {basis_name("cf", a, k): np.zeros(n_obj, dtype=np.float32) for a in ATOMS for k in KERNELS}
    offsets = np.arange(1, CF_HISTORY + 1)
    for b in range(0, n_obj, CF_BATCH):
        sl = slice(b, b + CF_BATCH)
        ob, tb = o[sl], target[sl]
        hist = ob[:, None] - offsets[None, :]
        inside = hist >= h.user_start[ob - 1][:, None]
        in_session = hist >= h.session_start[ob - 1][:, None]
        hist = np.where(inside, hist, 0)
        h_row = np.where(inside, item_row[hist], -1)
        valid = (h_row >= 0) & (tb >= 0)[:, None]
        sims = np.einsum("bkd,bd->bk", cf.item_unit[np.maximum(h_row, 0)], cf.item_unit[np.maximum(tb, 0)])
        sims = np.where(valid, sims, 0.0)
        weighted = sims[:, :, None] * atom_matrix[hist] * inside[:, :, None]
        age = np.where(inside, ref[sl][:, None] - h.ts[hist], 0).astype(np.float64)
        parts = {kernel: weighted[:, :k].sum(1) for kernel, k in ROW_KERNELS.items()}
        parts["sess"] = (weighted * in_session[:, :, None]).sum(1)
        for kernel, tau in TIME_KERNELS.items():
            parts[kernel] = (weighted * np.exp(-age / tau)[:, :, None]).sum(1)
        parts["all"] = weighted.sum(1)
        for kernel, part in parts.items():
            for j, a in enumerate(ATOMS):
                out[basis_name("cf", a, kernel)][sl] = part[:, j]
    return out


def degenerate_columns():
    return {basis_name(r, a, k) for r in ["u", "cf"] for a in ATOMS for k in RECENCY_KERNELS}


def basis_features(h, rel, cf):
    atoms = atom_values(h)
    session_from = h.session_start[h.objects - 1]
    out = {}
    for name in ["u", "ui", "ua", "ual"]:
        out.update(relation_basis(name, rel[name], atoms, session_from))
    out.update(cf_basis(cf, h, atoms))
    return {c: out[c] for c in basis_columns()}

In [ ]:
%%writefile /kaggle/working/code/rank/user_features.py
import numpy as np
import polars as pl

from . import config
from .basis import basis_features
from .cf import cf_features
from .windows import ALL, GroupIndex, WindowQuery

H = config.HOUR
D = config.DAY
WINDOWS = {"1h": H, "1d": D, "7d": 7 * D, "30d": 30 * D, "all": ALL}
TAUS = {"1h": H, "6h": 6 * H, "1d": D, "3d": 3 * D, "7d": 7 * D, "30d": 30 * D, "90d": 90 * D}
LISTEN_COLS = [
    "uid", "timestamp", "item_id", "is_organic", "played_ratio_pct", "track_length_seconds",
    "artist_id", "album_id", "listen_plus", "track_seen_n", "artist_seen_n",
]
DECAY_SPECS = {
    "u": {"sums": {"cnt": "one", "rec_cnt": "rec", "rec_lp": "lp_rec"}, "rates": {"rec_lp_rate": ("rec_lp", "rec_cnt")}, "hidden": ["rec_lp"]},
    "ui": {"sums": {"cnt": "one", "org_cnt": "org", "lp": "lp"}, "rates": {"lp_rate": ("lp", "cnt")}, "hidden": ["lp"]},
    "ua": {"sums": {"cnt": "one", "org_cnt": "org", "lp": "lp"}, "rates": {"lp_rate": ("lp", "cnt")}, "hidden": ["lp"]},
    "ual": {"sums": {"cnt": "one", "lp": "lp"}, "rates": {"lp_rate": ("lp", "cnt")}, "hidden": ["lp"]},
}
DECAY_MIN_MASS = 0.05


def rate(num, den):
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(den > 0, num / den, np.nan).astype(np.float32)


def dense_pairs(*code_arrays):
    codes = np.concatenate(code_arrays)
    _, inverse = np.unique(codes, return_inverse=True)
    out, pos = [], 0
    for arr in code_arrays:
        out.append(inverse[pos:pos + len(arr)])
        pos += len(arr)
    return out


def pair_code(uid, other):
    return (uid.astype(np.int64) << 32) | other.astype(np.int64)


class History:
    def __init__(self, chunk, t_end):
        chunk = chunk.select(LISTEN_COLS).sort("uid", "timestamp", maintain_order=True)
        self.n = chunk.height
        self.row = np.arange(self.n, dtype=np.int64)
        self.uid = chunk["uid"].to_numpy().astype(np.int64)
        self.ts = chunk["timestamp"].to_numpy().astype(np.int64)
        self.item = chunk["item_id"].to_numpy().astype(np.int64)
        self.has_artist = chunk["artist_id"].is_not_null().to_numpy()
        self.has_album = chunk["album_id"].is_not_null().to_numpy()
        self.artist = chunk["artist_id"].fill_null(0).to_numpy().astype(np.int64)
        self.album = chunk["album_id"].fill_null(0).to_numpy().astype(np.int64)
        self.org = chunk["is_organic"].to_numpy().astype(np.float64)
        self.rec = 1.0 - self.org
        self.ratio = chunk["played_ratio_pct"].to_numpy().astype(np.float64)
        self.length = chunk["track_length_seconds"].fill_null(0).to_numpy().astype(np.float64)
        self.lp = chunk["listen_plus"].to_numpy().astype(np.float64)
        self.new_track = (chunk["track_seen_n"].to_numpy() == 0).astype(np.float64)
        self.new_artist = (chunk["artist_seen_n"].fill_null(1).to_numpy() == 0).astype(np.float64)

        self.has_prev = np.r_[False, self.uid[1:] == self.uid[:-1]]
        self.prev_ts = np.r_[0, self.ts[:-1]]
        self.user_start = np.maximum.accumulate(np.where(~self.has_prev, self.row, 0))
        played_s = self.length * np.minimum(self.ratio, 100) / 100
        new_session = ~self.has_prev | (self.ts - self.prev_ts - played_s > config.SESSION_IDLE)
        self.session_start = np.maximum.accumulate(np.where(new_session, self.row, 0))
        self.days_before_end = (t_end - self.ts) // D
        is_obj = (self.org == 0) & (self.days_before_end < config.OBJECT_DAYS) & self.has_prev
        self.objects = np.flatnonzero(is_obj)
        self.ref = self.prev_ts[self.objects]

    def values(self):
        return {
            "one": np.ones(self.n), "rec": self.rec, "org": self.org, "lp": self.lp,
            "lp_rec": self.lp * self.rec, "lp_org": self.lp * self.org,
            "new_track": self.new_track, "new_artist": self.new_artist,
            "lp_new_artist": self.lp * self.new_artist, "len_lp": self.length * self.lp,
        }


class Relation:
    def __init__(self, history, rows, group_rows, group_objects, valid=None):
        h = history
        self.rows = rows
        self.index = GroupIndex(group_rows, h.ts[rows], row=h.row[rows])
        self.query = WindowQuery(self.index, group_objects, h.ref, end_row=h.objects, valid=valid)

    def cumsum(self, values):
        return self.index.cumsum(values[self.rows])

    def window_sum(self, values, window):
        return self.query.masked(self.query.sum(self.cumsum(values), WINDOWS[window])).astype(np.float32)

    def decayed_sums(self, values, tau):
        prefixes = self.index.decay_prefixes({k: v[self.rows] for k, v in values.items()}, tau)
        return {k: self.query.decayed(p) for k, p in prefixes.items()}

    def row_sum(self, values, k):
        return self.query.masked(self.query.row_sum(self.cumsum(values), k)).astype(np.float32)

    def since_row(self, values, from_row):
        return self.query.masked(self.query.since_row(self.cumsum(values), from_row)).astype(np.float32)

    def last_decayed(self, values, tau):
        q = self.query
        last = q.last(self.index.sorted_values(values[self.rows]), fill=0.0)
        age = q.ref_ts - q.last_ts()
        weight = np.where(np.isnan(age), 0.0, np.exp(-np.nan_to_num(age) / tau))
        return q.masked(last * weight).astype(np.float32)

    def recency_hours(self):
        return self.query.masked((self.query.ref_ts - self.query.last_ts()) / H).astype(np.float32)


def build_relations(h):
    o = h.objects
    everything = h.row
    relations = {"u": Relation(h, everything, h.uid, h.uid[o])}
    track_rows, track_obj = dense_pairs(pair_code(h.uid, h.item), pair_code(h.uid[o], h.item[o]))
    relations["ui"] = Relation(h, everything, track_rows, track_obj)
    for name, ent, has in [("ua", h.artist, h.has_artist), ("ual", h.album, h.has_album)]:
        rows = np.flatnonzero(has)
        g_rows, g_obj = dense_pairs(pair_code(h.uid[rows], ent[rows]), pair_code(h.uid[o], ent[o]))
        relations[name] = Relation(h, rows, g_rows, g_obj, valid=has[o])
    return relations


def user_window_features(h, rel, v):
    u = rel["u"]
    f = {}
    for w in ["1d", "7d", "30d"]:
        f[f"u_cnt_{w}"] = u.window_sum(v["one"], w)
    for w in ["1h", "1d", "7d", "30d", "all"]:
        f[f"u_rec_cnt_{w}"] = u.window_sum(v["rec"], w)
        f[f"u_rec_lp_rate_{w}"] = rate(u.window_sum(v["lp_rec"], w), f[f"u_rec_cnt_{w}"])
    n30 = f["u_cnt_30d"]
    org30 = u.window_sum(v["org"], "30d")
    new_artist30 = u.window_sum(v["new_artist"], "30d")
    f["u_org_share_30d"] = rate(org30, n30)
    f["u_org_lp_rate_30d"] = rate(u.window_sum(v["lp_org"], "30d"), org30)
    f["u_new_track_share_30d"] = rate(u.window_sum(v["new_track"], "30d"), n30)
    f["u_new_artist_share_30d"] = rate(new_artist30, n30)
    f["u_new_artist_lp_rate_30d"] = rate(u.window_sum(v["lp_new_artist"], "30d"), new_artist30)
    f["u_mean_len_lp_30d"] = rate(u.window_sum(v["len_lp"], "30d"), u.window_sum(v["lp"], "30d"))
    f["u_tenure_days"] = ((h.ref - u.query.first_ts()) / D).astype(np.float32)
    return f


def pair_window_features(h, rel, v):
    f = {}
    ui = rel["ui"]
    for w in ["1h", "1d", "7d", "30d", "all"]:
        f[f"ui_cnt_{w}"] = ui.window_sum(v["one"], w)
    for w in ["30d", "all"]:
        f[f"ui_org_cnt_{w}"] = ui.window_sum(v["org"], w)
        f[f"ui_lp_rate_{w}"] = rate(ui.window_sum(v["lp"], w), f[f"ui_cnt_{w}"])
    f["ui_last_ratio"] = ui.query.last(ui.index.sorted_values(h.ratio)).astype(np.float32)
    f["ui_recency_h"] = ui.recency_hours()

    ua = rel["ua"]
    for w in ["1d", "7d", "30d", "all"]:
        f[f"ua_cnt_{w}"] = ua.window_sum(v["one"], w)
    for w in ["30d", "all"]:
        f[f"ua_lp_rate_{w}"] = rate(ua.window_sum(v["lp"], w), f[f"ua_cnt_{w}"])
        f[f"ua_org_cnt_{w}"] = ua.window_sum(v["org"], w)
    f["ua_share_30d"] = rate(f["ua_cnt_30d"], rel["u"].window_sum(v["one"], "30d"))
    f["ua_recency_h"] = ua.recency_hours()

    ual = rel["ual"]
    for w in ["30d", "all"]:
        f[f"ual_cnt_{w}"] = ual.window_sum(v["one"], w)
    f["ual_lp_rate_all"] = rate(ual.window_sum(v["lp"], "all"), f["ual_cnt_all"])
    f["ual_recency_h"] = ual.recency_hours()
    return f


def decay_features(rel, v, taus):
    f = {}
    for name, spec in DECAY_SPECS.items():
        for tau_name in taus:
            sums = rel[name].decayed_sums({key: v[source] for key, source in spec["sums"].items()}, TAUS[tau_name])
            for key, value in sums.items():
                if key not in spec["hidden"]:
                    f[f"d_{name}_{key}_{tau_name}"] = value
            for key, (num, den) in spec["rates"].items():
                f[f"d_{name}_{key}_{tau_name}"] = np.where(sums[den] >= DECAY_MIN_MASS, sums[num] / np.maximum(sums[den], 1e-12), np.nan).astype(np.float32)
    return f


def explicit_features(h, explicit):
    o, q = h.objects, h.ref
    names = ["u_like_cnt_30d", "u_dislike_cnt_30d", "ui_like_state", "ui_dislike_state", "ua_liked_tracks", "ua_disliked_tracks"]
    if explicit.height == 0:
        return {c: np.zeros(len(o), dtype=np.float32) for c in names}
    e_uid = explicit["uid"].to_numpy().astype(np.int64)
    e_ts = explicit["timestamp"].to_numpy().astype(np.int64)
    e_item = explicit["item_id"].to_numpy().astype(np.int64)
    e_has_artist = explicit["artist_id"].is_not_null().to_numpy()
    e_artist = explicit["artist_id"].fill_null(0).to_numpy().astype(np.int64)
    e = {c: explicit[c].to_numpy().astype(np.float64) for c in ["is_like", "is_dislike", "like_delta", "dislike_delta"]}
    f = {}

    idx = GroupIndex(e_uid, e_ts)
    wq = WindowQuery(idx, h.uid[o], q, end_ts=q)
    f["u_like_cnt_30d"] = wq.sum(idx.cumsum(e["is_like"]), WINDOWS["30d"])
    f["u_dislike_cnt_30d"] = wq.sum(idx.cumsum(e["is_dislike"]), WINDOWS["30d"])

    g_e, g_o = dense_pairs(pair_code(e_uid, e_item), pair_code(h.uid[o], h.item[o]))
    idx = GroupIndex(g_e, e_ts)
    wq = WindowQuery(idx, g_o, q, end_ts=q)
    f["ui_like_state"] = np.clip(wq.sum(idx.cumsum(e["like_delta"]), ALL), 0, 1)
    f["ui_dislike_state"] = np.clip(wq.sum(idx.cumsum(e["dislike_delta"]), ALL), 0, 1)

    er = np.flatnonzero(e_has_artist)
    g_e, g_o = dense_pairs(pair_code(e_uid[er], e_artist[er]), pair_code(h.uid[o], h.artist[o]))
    idx = GroupIndex(g_e, e_ts[er])
    wq = WindowQuery(idx, g_o, q, end_ts=q, valid=h.has_artist[o])
    f["ua_liked_tracks"] = wq.masked(np.maximum(wq.sum(idx.cumsum(e["like_delta"][er]), ALL), 0)).astype(np.float32)
    f["ua_disliked_tracks"] = wq.masked(np.maximum(wq.sum(idx.cumsum(e["dislike_delta"][er]), ALL), 0)).astype(np.float32)
    return f


def context_features(h):
    o, q = h.objects, h.ref
    p = o - 1
    session_start = h.session_start
    lp_cs = np.r_[0.0, np.cumsum(h.lp)]
    pos_prev = p - session_start[p]
    f = {
        "c_pos_in_session": (pos_prev + 1).astype(np.float32),
        "c_session_age_min": ((q - h.ts[session_start[p]]) / 60).astype(np.float32),
        "c_session_lp_rate": ((lp_cs[o] - lp_cs[session_start[p]]) / (pos_prev + 1)).astype(np.float32),
    }
    for k in [1, 3, 10]:
        lo = np.maximum(o - k, h.user_start[p])
        f[f"c_prev{k}_lp_rate"] = ((lp_cs[o] - lp_cs[lo]) / (o - lo)).astype(np.float32)
    f["c_prev1_ratio"] = h.ratio[p].astype(np.float32)
    f["c_prev1_organic"] = h.org[p].astype(np.float32)
    f["c_prev_same_artist"] = ((h.artist[p] == h.artist[o]) & h.has_artist[o]).astype(np.float32)
    f["c_hour"] = ((q % D) // H).astype(np.float32)
    f["c_weekday"] = ((q // D) % 7).astype(np.float32)
    f["i_len"] = h.length[o].astype(np.float32)
    return f


def canary_features(h):
    o = h.objects
    next_ratio = np.r_[h.ratio[1:], np.nan]
    same_next = np.r_[h.uid[1:] == h.uid[:-1], False]
    return {
        "canary_next_ratio": np.where(same_next, next_ratio, np.nan)[o].astype(np.float32),
        "canary_gap_to_prev": (h.ts[o] - h.ref).astype(np.float32),
    }


def object_ids(h):
    o = h.objects
    ids = pl.DataFrame({
        "uid": h.uid[o].astype(np.uint32),
        "timestamp": h.ts[o].astype(np.uint32),
        "ref_ts": h.ref.astype(np.uint32),
        "item_id": h.item[o].astype(np.uint32),
        "artist_id": pl.Series(h.artist[o].astype(np.uint32)).set(pl.Series(~h.has_artist[o]), None),
        "album_id": pl.Series(h.album[o].astype(np.uint32)).set(pl.Series(~h.has_album[o]), None),
        "days_before_end": h.days_before_end[o].astype(np.int16),
        "label": h.lp[o].astype(np.int8),
    })
    split = (
        pl.when(pl.col("days_before_end") < config.TEST_DAYS).then(pl.lit("test"))
        .when(pl.col("days_before_end") < config.TEST_DAYS + config.VAL_DAYS).then(pl.lit("val"))
        .otherwise(pl.lit("train"))
    )
    return ids.with_columns(split.alias("split"))


def user_part_features(chunk, explicit, t_end, cf=None, taus=tuple(TAUS), basis=False, canary=False):
    h = History(chunk, t_end)
    v = h.values()
    rel = build_relations(h)
    f = {}
    f.update(user_window_features(h, rel, v))
    f.update(pair_window_features(h, rel, v))
    f.update(explicit_features(h, explicit))
    f.update(context_features(h))
    if taus:
        f.update(decay_features(rel, v, taus))
    if cf is not None:
        f.update(cf_features(cf, h.uid, h.item, h.lp, h.objects, h.user_start))
    if basis:
        f.update(basis_features(h, rel, cf))
    if canary:
        f.update(canary_features(h))
    return object_ids(h).hstack(pl.DataFrame({k: np.asarray(x, dtype=np.float32) for k, x in f.items()}))

In [ ]:
%%writefile /kaggle/working/code/rank/entity_features.py
import gc

import numpy as np
import polars as pl

from . import config
from .windows import GroupIndex, WindowQuery
from .user_features import WINDOWS

D = config.DAY


def smooth(num, den, prior):
    return ((num + config.SMOOTH_M * prior) / (den + config.SMOOTH_M)).astype(np.float32)


def entity_window_sums(group, ts, values, q_group, day_start, windows, valid=None):
    idx = GroupIndex(group, ts)
    wq = WindowQuery(idx, q_group, day_start, end_ts=day_start, valid=valid)
    out = {}
    for name, v in values.items():
        csum = idx.cumsum(v)
        for w in windows[name]:
            out[(name, w)] = wq.sum(csum, WINDOWS[w])
        del csum
    first = wq.first_ts()
    del idx, wq
    gc.collect()
    return out, first


def entity_features(listens, explicit, objects, prior):
    day_start = (objects["ref_ts"].to_numpy().astype(np.int64) // D) * D
    data = listens.select("item_id", "artist_id", "album_id", "timestamp", "is_organic", "listen_plus").collect()
    ts = data["timestamp"].to_numpy()
    rec = (data["is_organic"].to_numpy() == 0).astype(np.uint8)
    lp_rec = (data["listen_plus"].to_numpy() & (rec == 1)).astype(np.uint8)
    one = np.ones(len(ts), dtype=np.uint8)
    f = {}

    sums, first = entity_window_sums(
        data["item_id"].to_numpy(), ts, {"one": one, "rec": rec, "lp": lp_rec},
        objects["item_id"].to_numpy(), day_start,
        {"one": ["1d", "7d", "30d", "all"], "rec": ["7d", "30d", "all"], "lp": ["7d", "30d", "all"]},
    )
    for w in ["1d", "7d", "30d", "all"]:
        f[f"i_cnt_{w}"] = sums[("one", w)]
    for w in ["7d", "30d", "all"]:
        f[f"i_rec_cnt_{w}"] = sums[("rec", w)]
        f[f"i_rec_lp_smooth_{w}"] = smooth(sums[("lp", w)], sums[("rec", w)], prior)
    f["i_trend_7d_30d"] = (f["i_cnt_7d"] / (f["i_cnt_30d"] + 1)).astype(np.float32)
    f["i_age_days"] = np.nan_to_num((day_start - first) / D, nan=0.0).astype(np.float32)

    sums, _ = entity_window_sums(
        explicit["item_id"].to_numpy(), explicit["timestamp"].to_numpy(),
        {"like": explicit["is_like"].to_numpy(), "dislike": explicit["is_dislike"].to_numpy()},
        objects["item_id"].to_numpy(), day_start, {"like": ["all"], "dislike": ["all"]},
    )
    f["i_like_rate_all"] = (sums[("like", "all")] / (f["i_cnt_all"] + 10)).astype(np.float32)
    f["i_dislike_rate_all"] = (sums[("dislike", "all")] / (f["i_cnt_all"] + 10)).astype(np.float32)

    for prefix, col, cnt_windows, lp_windows in [
        ("a", "artist_id", ["7d", "30d"], ["30d", "all"]),
        ("al", "album_id", ["30d"], ["all"]),
    ]:
        keep = data[col].is_not_null().to_numpy()
        valid = objects[col].is_not_null().to_numpy()
        sums, _ = entity_window_sums(
            data[col].to_numpy()[keep], ts[keep], {"one": one[keep], "rec": rec[keep], "lp": lp_rec[keep]},
            objects[col].fill_null(0).to_numpy(), day_start,
            {"one": cnt_windows, "rec": lp_windows, "lp": lp_windows}, valid=valid,
        )
        for w in cnt_windows:
            f[f"{prefix}_cnt_{w}"] = np.where(valid, sums[("one", w)], np.nan).astype(np.float32)
        for w in lp_windows:
            f[f"{prefix}_rec_lp_smooth_{w}"] = np.where(valid, smooth(sums[("lp", w)], sums[("rec", w)], prior), np.nan).astype(np.float32)
        del keep, sums
        gc.collect()
    del data, ts, rec, lp_rec, one
    gc.collect()
    return pl.DataFrame({k: np.asarray(v, dtype=np.float32) for k, v in f.items()})

In [ ]:
%%writefile /kaggle/working/code/rank/features.py
import gc
import hashlib
import json
import shutil
import time
from pathlib import Path

import polars as pl

from . import config
from .cf import fit_item_factors
from .entity_features import entity_features
from .user_features import TAUS, WINDOWS, user_part_features

D = config.DAY
ID_COLS = ["obj_id", "uid", "timestamp", "ref_ts", "item_id", "artist_id", "album_id", "days_before_end", "split", "label"]
GROUPS = {
    "user": ("u_",),
    "item": ("i_", "a_", "al_"),
    "pair_track": ("ui_",),
    "pair_artist_album": ("ua_", "ual_"),
    "context": ("c_",),
    "cross": ("x_",),
    "decay": ("d_",),
    "cf": ("cf_",),
    "basis": ("b_",),
}
FEATURE_MODULES = [
    "__init__.py", "config.py", "prepare.py", "windows.py", "cf.py", "basis.py",
    "user_features.py", "entity_features.py", "features.py",
]
WINDOW_BASES = [
    "u_cnt", "u_rec_cnt", "u_rec_lp_rate", "ui_cnt", "ui_org_cnt", "ui_lp_rate",
    "ua_cnt", "ua_org_cnt", "ua_lp_rate", "ual_cnt", "ual_lp_rate",
]


def code_hash():
    digest = hashlib.sha256()
    for name in FEATURE_MODULES:
        digest.update((Path(__file__).parent / name).read_bytes())
    return digest.hexdigest()[:16]


def find_cached(root, taus=tuple(TAUS), basis=False):
    for meta_path in sorted(Path(root).rglob("meta.json")):
        meta = json.loads(meta_path.read_text())
        ok = (
            meta.get("code_hash") == code_hash()
            and meta.get("max_parts") is None
            and meta.get("max_ts") is None
            and set(taus) <= set(meta.get("taus", []))
            and (meta.get("basis") or not basis)
            and all((meta_path.parent / f"{s}.parquet").exists() for s in config.SPLITS)
        )
        if ok:
            return meta_path.parent
    return None


def dataset_end(events_path):
    return int(pl.scan_parquet(events_path).select(pl.col("timestamp").max()).collect().item())


def load_explicit(events_path, artist_of, max_ts=None):
    lf = pl.scan_parquet(events_path).filter(pl.col("event_type") != "listen")
    if max_ts is not None:
        lf = lf.filter(pl.col("timestamp") < max_ts)
    event = pl.col("event_type").cast(pl.Utf8)
    return (
        lf.select("uid", "item_id", "timestamp", "event_type")
        .with_columns(
            (event == "like").cast(pl.Int8).alias("is_like"),
            (event == "dislike").cast(pl.Int8).alias("is_dislike"),
            pl.when(event == "like").then(1).when(event == "unlike").then(-1).otherwise(0).cast(pl.Int8).alias("like_delta"),
            pl.when(event == "dislike").then(1).when(event == "undislike").then(-1).otherwise(0).cast(pl.Int8).alias("dislike_delta"),
        )
        .drop("event_type")
        .collect()
        .join(artist_of, on="item_id", how="left")
        .sort("uid", "timestamp", maintain_order=True)
    )


def add_cross(df):
    return df.with_columns(
        pl.when(pl.col("ui_cnt_all") > 0).then(0).when(pl.col("ua_cnt_all") > 0).then(1).otherwise(2).cast(pl.Float32).alias("x_novelty"),
        (pl.col("u_rec_lp_rate_30d") - pl.col("i_rec_lp_smooth_30d")).alias("x_user_minus_item"),
        (pl.col("ui_lp_rate_all") - pl.col("u_rec_lp_rate_all")).alias("x_pair_vs_user"),
        (pl.col("ua_lp_rate_all") - pl.col("u_rec_lp_rate_all")).alias("x_artist_vs_user"),
        (pl.col("i_len") / pl.col("u_mean_len_lp_30d")).alias("x_len_ratio"),
    )


def feature_columns(df):
    return [c for c in df.columns if c not in ID_COLS and not c.startswith("canary_")]


def window_features(cols):
    names = {f"{base}_{w}" for base in WINDOW_BASES for w in WINDOWS}
    return [c for c in cols if c in names]


def decay_features_for(cols, taus):
    suffixes = tuple(f"_{t}" for t in taus)
    return [c for c in cols if c.startswith("d_") and c.endswith(suffixes)]


def history_prior(listens, t_end):
    return float(
        listens.filter(pl.col("is_organic") == 0, pl.col("timestamp") <= t_end - config.OBJECT_DAYS * D)
        .select(pl.col("listen_plus").mean())
        .collect()
        .item()
    )


def cf_cutoff(t_end):
    return t_end - config.OBJECT_DAYS * D


def scan_listens_dir(listens_dir, max_ts=None):
    lf = pl.scan_parquet(sorted(Path(listens_dir).glob("part_*.parquet")))
    return lf.filter(pl.col("timestamp") < max_ts) if max_ts is not None else lf


def build_features(listens_dir, events_path, mappings_dir, out_dir, t_end=None, max_ts=None, max_parts=None,
                   prior=None, cf="fit", taus=tuple(TAUS), basis=False, canary=False):
    from .prepare import load_mappings

    started = time.time()
    out_dir = Path(out_dir)
    if out_dir.exists():
        shutil.rmtree(out_dir)
    (out_dir / "parts").mkdir(parents=True)
    parts = sorted(Path(listens_dir).glob("part_*.parquet"))[:max_parts]

    listens = scan_listens_dir(listens_dir, max_ts)
    t_end = t_end if t_end is not None else dataset_end(events_path)
    prior = prior if prior is not None else history_prior(listens, t_end)
    if cf == "fit":
        cf = fit_item_factors(listens, cf_cutoff(t_end))
    artist_of, _ = load_mappings(events_path, mappings_dir)
    explicit = load_explicit(events_path, artist_of, max_ts=max_ts)
    print(f"t_end {t_end}, prior {prior:.4f}, explicit events {explicit.height}, parts {len(parts)}, taus {list(taus)}, basis {basis}")

    offset = 0
    for path in parts:
        lf = pl.scan_parquet(path)
        chunk = (lf.filter(pl.col("timestamp") < max_ts) if max_ts is not None else lf).collect()
        if chunk.height == 0:
            continue
        part_explicit = explicit.filter(pl.col("uid").is_between(chunk["uid"].min(), chunk["uid"].max()))
        feats = user_part_features(chunk, part_explicit, t_end, cf=cf, taus=taus, basis=basis, canary=canary)
        feats = feats.with_columns((pl.int_range(feats.height, dtype=pl.UInt32) + offset).alias("obj_id"))
        offset += feats.height
        feats.write_parquet(out_dir / "parts" / path.name)
        print(f"{path.name}: rows {chunk.height}, objects {feats.height}, {time.time() - started:.0f}s")
        del chunk, feats, part_explicit
        gc.collect()

    user_feats = pl.scan_parquet(out_dir / "parts" / "*.parquet")
    objects = user_feats.select("obj_id", "item_id", "artist_id", "album_id", "ref_ts").collect().sort("obj_id")
    ent = entity_features(listens, explicit, objects, prior).with_columns(objects["obj_id"])
    print(f"entity features: {ent.width - 1} columns, {time.time() - started:.0f}s")
    del objects, explicit
    gc.collect()

    for split in config.SPLITS:
        df = add_cross(user_feats.filter(pl.col("split") == split).collect().join(ent, on="obj_id", how="left"))
        df.write_parquet(out_dir / f"{split}.parquet")
        rate = df["label"].mean() if df.height else float("nan")
        print(f"{split}: {df.height} objects, positive rate {rate:.4f}")
        cols = feature_columns(df)
        del df
        gc.collect()

    meta = {
        "t_end": t_end,
        "prior": prior,
        "max_ts": max_ts,
        "max_parts": max_parts,
        "code_hash": code_hash(),
        "cf_cutoff_ts": cf.cutoff_ts if cf is not None else None,
        "taus": list(taus),
        "basis": basis,
        "features": cols,
        "groups": {g: [c for c in cols if c.startswith(prefixes)] for g, prefixes in GROUPS.items()},
        "window_features": window_features(cols),
        "seconds": round(time.time() - started, 1),
    }
    (out_dir / "meta.json").write_text(json.dumps(meta, indent=2))
    shutil.rmtree(out_dir / "parts")
    print(f"done: {len(cols)} features -> {out_dir}, {meta['seconds']}s")
    return meta

In [ ]:
%%writefile /kaggle/working/code/rank/metrics.py
import numpy as np
import polars as pl
from sklearn.metrics import log_loss, roc_auc_score

from . import config

NOVELTY = {0: "repeat_track", 1: "new_track_known_artist", 2: "new_artist"}
METRICS = ("logloss", "auc", "gauc", "ndcg@10")


def per_user_auc(frame):
    return (
        frame.with_columns(pl.col("pred").rank("average").over("uid").alias("r"))
        .group_by("uid")
        .agg(
            pl.len().alias("n"),
            pl.col("label").sum().alias("pos"),
            pl.col("r").filter(pl.col("label") == 1).sum().alias("rank_pos"),
        )
        .filter((pl.col("pos") > 0) & (pl.col("pos") < pl.col("n")))
        .with_columns(
            ((pl.col("rank_pos") - pl.col("pos") * (pl.col("pos") + 1) / 2) / (pl.col("pos") * (pl.col("n") - pl.col("pos")))).alias("auc")
        )
        .select("uid", "n", "auc")
    )


def per_group_ndcg(frame, k=10):
    position = pl.int_range(1, pl.len() + 1).cast(pl.Float64)
    gain = 1.0 / (position + 1).log(2)
    return (
        frame.group_by("uid", "day")
        .agg(
            (pl.col("label").sort_by("pred", descending=True).cast(pl.Float64) * gain).head(k).sum().alias("dcg"),
            (pl.col("label").sort(descending=True).cast(pl.Float64) * gain).head(k).sum().alias("idcg"),
        )
        .filter(pl.col("idcg") > 0)
        .with_columns((pl.col("dcg") / pl.col("idcg")).alias("ndcg"))
        .select("uid", "day", "ndcg")
    )


class Scorer:
    def __init__(self, frame):
        self.frame = frame
        self.y = frame["label"].to_numpy()
        self.p = np.clip(frame["pred"].to_numpy(), 1e-6, 1 - 1e-6)
        self.uid = frame["uid"].to_numpy()
        self.users = per_user_auc(frame)
        self.groups = per_group_ndcg(frame)

    def score(self, user_weights=None):
        if user_weights is None:
            w = None
            users_w = self.users["n"].to_numpy().astype(np.float64)
            groups_w = np.ones(self.groups.height)
        else:
            w = user_weights[self.uid]
            users_w = self.users["n"].to_numpy() * user_weights[self.users["uid"].to_numpy()]
            groups_w = user_weights[self.groups["uid"].to_numpy()].astype(np.float64)
        has_both = len(np.unique(self.y)) > 1
        return {
            "n": int(len(self.y)),
            "logloss": float(log_loss(self.y, self.p, sample_weight=w, labels=[0, 1])),
            "auc": float(roc_auc_score(self.y, self.p, sample_weight=w)) if has_both else float("nan"),
            "gauc": float(np.average(self.users["auc"].to_numpy(), weights=users_w)) if users_w.sum() > 0 else float("nan"),
            "ndcg@10": float(np.average(self.groups["ndcg"].to_numpy(), weights=groups_w)) if groups_w.sum() > 0 else float("nan"),
        }


def prepare_frame(df, pred):
    return pl.DataFrame({
        "uid": df["uid"].cast(pl.Int64),
        "day": df["days_before_end"],
        "label": df["label"].cast(pl.Int32),
        "pred": np.asarray(pred, dtype=np.float64),
        "novelty": df["x_novelty"].cast(pl.Int32),
    })


def evaluate_slices(df, pred):
    frame = prepare_frame(df, pred)
    rows = [{"slice": "all", **Scorer(frame).score()}]
    for code, name in NOVELTY.items():
        part = frame.filter(pl.col("novelty") == code)
        if part.height:
            rows.append({"slice": name, **Scorer(part).score()})
    return rows


def bootstrap(df, preds, n_boot=100, seed=config.SEED):
    scorers = {name: Scorer(prepare_frame(df, p)) for name, p in preds.items()}
    uids = np.unique(df["uid"].to_numpy()).astype(np.int64)
    rng = np.random.default_rng(seed)
    samples = {name: [] for name in preds}
    for _ in range(n_boot):
        counts = np.bincount(rng.integers(0, len(uids), len(uids)), minlength=len(uids)).astype(np.float64)
        weights = np.zeros(uids.max() + 1)
        weights[uids] = counts
        for name, scorer in scorers.items():
            samples[name].append(scorer.score(weights))
    return samples


def interval(values):
    values = np.asarray(values)
    return {"mean": float(values.mean()), "lo": float(np.quantile(values, 0.025)), "hi": float(np.quantile(values, 0.975))}


def summarize(samples):
    return {name: {m: interval([r[m] for r in runs]) for m in METRICS} for name, runs in samples.items()}


def paired_delta(samples, a, b):
    return {m: interval([x[m] - y[m] for x, y in zip(samples[a], samples[b])]) for m in METRICS}

In [ ]:
%%writefile /kaggle/working/code/rank/leakage.py
import json
from pathlib import Path

import numpy as np
import polars as pl
from sklearn.metrics import roc_auc_score

from . import config
from .cf import fit_item_factors
from .features import build_features, cf_cutoff, scan_listens_dir


def load_all(features_dir):
    return pl.concat([pl.read_parquet(Path(features_dir) / f"{s}.parquet") for s in config.SPLITS], how="vertical")


def compare(full, trunc, columns):
    report = {}
    for c in columns:
        a = full[c].to_numpy().astype(np.float64)
        b = trunc[c].to_numpy().astype(np.float64)
        both_nan = np.isnan(a) & np.isnan(b)
        close = np.isclose(a, b, rtol=1e-5, atol=1e-6)
        report[c] = int((~(both_nan | close)).sum())
    return report


def single_feature_auc(df, columns):
    y = df["label"].to_numpy()
    out = {}
    for c in columns:
        x = df[c].to_numpy().astype(np.float64)
        ok = ~np.isnan(x)
        if ok.sum() > 100 and len(np.unique(y[ok])) > 1:
            auc = roc_auc_score(y[ok], x[ok])
            out[c] = float(max(auc, 1 - auc))
    return dict(sorted(out.items(), key=lambda kv: -kv[1]))


def check_leakage(listens_dir, events_path, mappings_dir, work_dir, t_end, prior, max_parts=1, cut_days_before_end=10, **feature_kwargs):
    work_dir = Path(work_dir)
    cut = t_end - cut_days_before_end * config.DAY
    cf = fit_item_factors(scan_listens_dir(listens_dir), cf_cutoff(t_end))
    common = dict(listens_dir=listens_dir, events_path=events_path, mappings_dir=mappings_dir, t_end=t_end, max_parts=max_parts, canary=True, prior=prior, cf=cf, **feature_kwargs)
    build_features(out_dir=work_dir / "leak_full", **common)
    build_features(out_dir=work_dir / "leak_trunc", max_ts=cut, **common)

    window = (pl.col("timestamp") >= cut - config.DAY) & (pl.col("timestamp") < cut)
    full = load_all(work_dir / "leak_full")
    trunc = load_all(work_dir / "leak_trunc")
    full_w = full.filter(window).sort("uid", "timestamp", "obj_id")
    trunc_w = trunc.filter(window).sort("uid", "timestamp", "obj_id")
    assert full_w.height == trunc_w.height, (full_w.height, trunc_w.height)

    meta = json.loads((work_dir / "leak_full" / "meta.json").read_text())
    canaries = [c for c in full.columns if c.startswith("canary_")]
    mismatches = compare(full_w, trunc_w, meta["features"] + canaries)
    real = {c: n for c, n in mismatches.items() if not c.startswith("canary_")}
    val = full.filter(pl.col("split") == "val")
    aucs = single_feature_auc(val, meta["features"] + canaries)
    report = {
        "cut_ts": cut,
        "objects_compared": full_w.height,
        "features_with_mismatch": {c: n for c, n in real.items() if n > 0},
        "canary_mismatches": {c: mismatches[c] for c in canaries},
        "passed": all(n == 0 for n in real.values()) and mismatches.get("canary_next_ratio", 0) > 0,
        "single_feature_auc_top10": dict(list({c: a for c, a in aucs.items() if not c.startswith("canary_")}.items())[:10]),
        "canary_auc": {c: aucs.get(c) for c in canaries},
    }
    return report

In [ ]:
%%writefile /kaggle/working/code/rank/train.py
import time

import numpy as np
import polars as pl
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

from . import config

LOGREG_FEATURES = [
    "u_rec_lp_rate_30d", "u_rec_lp_rate_1h", "u_org_share_30d", "i_rec_lp_smooth_30d", "i_len",
    "ui_cnt_all", "ui_lp_rate_all", "ui_recency_h", "ua_cnt_30d", "ua_lp_rate_all", "ua_share_30d",
    "ui_like_state", "ui_dislike_state", "c_prev1_lp_rate", "c_prev3_lp_rate", "c_session_lp_rate", "x_novelty",
]
SINGLE_FEATURES = ["u_rec_lp_rate_30d", "i_rec_lp_smooth_30d", "c_prev3_lp_rate"]
FINAL_ITERATION_FACTOR = 1.15


def catboost_params(task_type, iterations, learning_rate=0.08, depth=8, seed=config.SEED):
    params = dict(
        loss_function="Logloss",
        eval_metric="Logloss",
        iterations=iterations,
        learning_rate=learning_rate,
        depth=depth,
        l2_leaf_reg=3,
        random_seed=seed,
        task_type=task_type,
        metric_period=50,
        verbose=500,
    )
    if task_type == "CPU":
        params["thread_count"] = 4
    return params


def matrix(df, cols):
    return df.select(cols).to_numpy().astype(np.float32)


def signed_log(x):
    return np.sign(x) * np.log1p(np.abs(x))


class LinearModel:
    def __init__(self, cols):
        self.cols = cols

    def transform(self, df):
        x = matrix(df, self.cols)
        return signed_log(np.where(np.isnan(x), self.median, x))

    def fit(self, df):
        x = matrix(df, self.cols)
        self.median = np.nan_to_num(np.nanmedian(x, axis=0))
        x = signed_log(np.where(np.isnan(x), self.median, x))
        self.scaler = StandardScaler().fit(x)
        self.model = LogisticRegression(max_iter=500).fit(self.scaler.transform(x), df["label"].to_numpy())
        return self

    def predict(self, df):
        return self.model.predict_proba(self.scaler.transform(self.transform(df)))[:, 1]


def fit_catboost(train, val, cols, task_type, iterations, seed=config.SEED):
    model = CatBoostClassifier(**catboost_params(task_type, iterations, seed=seed), od_type="Iter", od_wait=200, use_best_model=True)
    started = time.time()
    model.fit(matrix(train, cols), train["label"].to_numpy(), eval_set=(matrix(val, cols), val["label"].to_numpy()))
    return model, time.time() - started


def refit_catboost(data, cols, task_type, best_iteration, seed=config.SEED):
    model = CatBoostClassifier(**catboost_params(task_type, int(best_iteration * FINAL_ITERATION_FACTOR), seed=seed))
    started = time.time()
    model.fit(matrix(data, cols), data["label"].to_numpy())
    return model, time.time() - started


def predict(model, df, cols):
    return model.predict_proba(matrix(df, cols))[:, 1]


def simple_baselines(train, val, test):
    trainval = pl.concat([train, val], how="vertical")
    val_preds = {"constant": np.full(val.height, train["label"].mean())}
    test_preds = {"constant": np.full(test.height, trainval["label"].mean())}
    for feat in SINGLE_FEATURES:
        val_preds[f"single:{feat}"] = LinearModel([feat]).fit(train).predict(val)
        test_preds[f"single:{feat}"] = LinearModel([feat]).fit(trainval).predict(test)
    val_preds["logreg"] = LinearModel(LOGREG_FEATURES).fit(train).predict(val)
    test_preds["logreg"] = LinearModel(LOGREG_FEATURES).fit(trainval).predict(test)
    return val_preds, test_preds

In [ ]:
%%writefile /kaggle/working/code/rank/experiments.py
import json
from pathlib import Path

import numpy as np
import polars as pl

from . import config
from .features import decay_features_for
from .metrics import bootstrap, evaluate_slices, paired_delta, summarize
from .train import fit_catboost, predict, refit_catboost, simple_baselines

REFERENCE = "L0+cf"
MATCHED_TAUS = ("1h", "1d", "7d", "30d", "90d")


def feature_sets(meta):
    cols = meta["features"]
    groups = meta["groups"]
    decay, cf, window = set(groups["decay"]), set(groups["cf"]), set(meta["window_features"])
    cols = [c for c in cols if c not in set(groups.get("basis", []))]
    l0 = [c for c in cols if c not in decay and c not in cf]
    l0_cf = l0 + [c for c in cols if c in cf]
    without_windows = [c for c in l0_cf if c not in window]
    return {
        "L0": l0,
        "L0+cf": l0_cf,
        "L1": without_windows + decay_features_for(cols, MATCHED_TAUS),
        "L1-wide": without_windows + [c for c in cols if c in decay],
        "L0+L1": cols,
    }


def slug(name):
    return name.replace("+", "_plus_").replace(":", "_")


def results_table(preds, df):
    rows = []
    for name, p in preds.items():
        rows.extend({"model": name, **r} for r in evaluate_slices(df, p))
    return pl.DataFrame(rows)


def ablation(train, val, cols, groups, reference_rows, task_type, iterations):
    rows = []
    base = {r["slice"]: r for r in reference_rows}
    for group, group_cols in groups.items():
        dropped = [c for c in group_cols if c in cols]
        if not dropped:
            continue
        keep = [c for c in cols if c not in dropped]
        model, seconds = fit_catboost(train, val, keep, task_type, iterations)
        for r in evaluate_slices(val, predict(model, val, keep)):
            b = base[r["slice"]]
            rows.append({
                "dropped_group": group, "n_dropped": len(dropped), "slice": r["slice"],
                "auc": r["auc"], "gauc": r["gauc"], "logloss": r["logloss"], "ndcg@10": r["ndcg@10"],
                "delta_auc": r["auc"] - b["auc"], "delta_gauc": r["gauc"] - b["gauc"], "delta_logloss": r["logloss"] - b["logloss"],
                "best_iter": model.get_best_iteration() + 1, "train_s": round(seconds, 1),
            })
        print(f"ablation {group}: {seconds:.0f}s")
    return pl.DataFrame(rows)


def importance_by_scale(importance, prefix, scale_names):
    rows = []
    for name, value in zip(importance["feature"], importance["importance"]):
        if not name.startswith(prefix):
            continue
        scale = next((s for s in scale_names if name.endswith(f"_{s}")), None)
        if scale is None:
            continue
        body = name[len(prefix):-len(scale) - 1]
        relation = body.split("_")[0]
        rows.append({"relation": relation, "feature_base": body, "scale": scale, "importance": value})
    return pl.DataFrame(rows)


def run_experiments(features_dir, out_dir, task_type="GPU", iterations=3000, n_boot=100, with_ablation=True, sets=None):
    features_dir, out_dir = Path(features_dir), Path(out_dir)
    for sub in ["models", "importance"]:
        (out_dir / sub).mkdir(parents=True, exist_ok=True)
    meta = json.loads((features_dir / "meta.json").read_text())
    sets = sets or feature_sets(meta)
    train, val, test = (pl.read_parquet(features_dir / f"{s}.parquet") for s in config.SPLITS)
    trainval = pl.concat([train, val], how="vertical")
    print(f"train {train.height}, val {val.height}, test {test.height}; sets {[(k, len(v)) for k, v in sets.items()]}")

    val_preds, test_preds = simple_baselines(train, val, test)
    records = {}
    for name, cols in sets.items():
        model, seconds = fit_catboost(train, val, cols, task_type, iterations)
        records[name] = {"features": len(cols), "best_iteration": model.get_best_iteration() + 1, "val_fit_s": round(seconds, 1)}
        val_preds[name] = predict(model, val, cols)
        print(f"{name}: {len(cols)} features, best iteration {records[name]['best_iteration']}, {seconds:.0f}s")

    val_table = results_table(val_preds, val)
    val_table.write_csv(out_dir / "val_metrics.csv")
    print(val_table.filter(pl.col("slice") == "all").select("model", "logloss", "auc", "gauc", "ndcg@10"))

    if with_ablation:
        reference_rows = val_table.filter(pl.col("model") == REFERENCE).drop("model").to_dicts()
        ablation(train, val, sets[REFERENCE], meta["groups"], reference_rows, task_type, iterations).write_csv(out_dir / "ablation_val.csv")

    importances = {}
    for name, cols in sets.items():
        model, seconds = refit_catboost(trainval, cols, task_type, records[name]["best_iteration"])
        records[name]["final_fit_s"] = round(seconds, 1)
        model.save_model(str(out_dir / "models" / f"{slug(name)}.cbm"))
        test_preds[name] = predict(model, test, cols)
        importances[name] = pl.DataFrame({"feature": cols, "importance": model.get_feature_importance()}).sort("importance", descending=True)
        importances[name].write_csv(out_dir / "importance" / f"{slug(name)}.csv")

    test_table = results_table(test_preds, test)
    test_table.write_csv(out_dir / "test_metrics.csv")
    print(test_table.filter(pl.col("slice") == "all").select("model", "logloss", "auc", "gauc", "ndcg@10"))

    boot_models = list(sets) + ["logreg"]
    samples = bootstrap(test, {k: test_preds[k] for k in boot_models}, n_boot=n_boot)
    deltas = {f"{name}_vs_{REFERENCE}": paired_delta(samples, name, REFERENCE) for name in boot_models if name != REFERENCE}

    if "L1-wide" in importances:
        importance_by_scale(importances["L1-wide"], "d_", meta["taus"]).write_csv(out_dir / "tau_importance.csv")
    importance_by_scale(importances[REFERENCE], "", ["1h", "1d", "7d", "30d", "all"]).filter(
        pl.col("feature_base").is_in([c.rsplit("_", 1)[0] for c in meta["window_features"]])
    ).write_csv(out_dir / "window_importance.csv")

    pred_cols = {f"pred_{slug(k)}": np.asarray(test_preds[k], dtype=np.float32) for k in boot_models}
    test.select("uid", "timestamp", "item_id", "days_before_end", "label", "x_novelty").with_columns(
        [pl.Series(k, v) for k, v in pred_cols.items()]
    ).write_parquet(out_dir / "test_predictions.parquet")

    summary = {
        "reference": REFERENCE,
        "rows": {"train": train.height, "val": val.height, "test": test.height},
        "feature_sets": {k: v for k, v in sets.items()},
        "records": records,
        "task_type": task_type,
        "n_boot": n_boot,
        "test_bootstrap_ci": summarize(samples),
        "test_paired_deltas": deltas,
    }
    (out_dir / "summary.json").write_text(json.dumps(summary, indent=2))
    return summary

In [ ]:
%%writefile /kaggle/working/code/rank/learned.py
import time

import numpy as np
import torch
from torch import nn

from . import config
from .basis import RELATIONS
from .train import matrix, signed_log

KERNEL_TEXT = {
    "r1": "last track", "r3": "last 3 tracks", "r10": "last 10 tracks",
    "t1h": "decay 1h", "t6h": "decay 6h", "t1d": "decay 1d", "t3d": "decay 3d",
    "t7d": "decay 7d", "t30d": "decay 30d", "t90d": "decay 90d", "all": "all history",
    "sess": "current session", "x1d": "last event, decay 1d", "x30d": "last event, decay 30d",
}
RELATION_TEXT = {"u": "any track", "ui": "same track", "ua": "same artist", "ual": "same album", "cf": "cf-similar tracks"}


def straight_through(hard, soft):
    return hard + soft - soft.detach()


def mean_entropy(p):
    return -(p * torch.log(p + 1e-9)).sum(-1).mean()


class Grammar(nn.Module):
    def __init__(self, n_count, n_rate, alphabet, init_scale=1.0):
        super().__init__()
        k = n_count + n_rate
        self.n_count = n_count
        self.alphabet = alphabet
        n_rel, n_atoms, n_kernels = alphabet.shape
        self.relation = nn.Parameter(init_scale * torch.randn(k, n_rel))
        self.kernel = nn.Parameter(init_scale * torch.randn(k, n_kernels))
        self.numerator = nn.Parameter(init_scale * torch.randn(k, n_atoms))
        self.denominator = nn.Parameter(init_scale * torch.randn(k, n_atoms))
        self.hard = False

    def weights(self):
        relation = torch.softmax(self.relation, -1)
        kernel = torch.softmax(self.kernel, -1)
        numerator = torch.sigmoid(self.numerator)
        extra = torch.sigmoid(self.denominator)
        if self.hard:
            relation = straight_through(nn.functional.one_hot(relation.argmax(-1), relation.shape[-1]).to(relation.dtype), relation)
            kernel = straight_through(nn.functional.one_hot(kernel.argmax(-1), kernel.shape[-1]).to(kernel.dtype), kernel)
            numerator = straight_through((numerator >= 0.5).to(numerator.dtype), numerator)
            extra = straight_through((extra >= 0.5).to(extra.dtype), extra)
        denominator = numerator + (1 - numerator) * extra
        return relation, kernel, numerator, denominator

    def forward(self, basis):
        relation, kernel, numerator, denominator = self.weights()
        mixed = torch.einsum("brem,kr,km->bke", basis, relation, kernel)
        top = (mixed * numerator).sum(-1)
        bottom = (mixed * denominator).sum(-1)
        counts = torch.log1p(top[:, :self.n_count].clamp(min=0))
        rates = (top[:, self.n_count:] + 0.5) / (bottom[:, self.n_count:].clamp(min=0) + 1)
        return torch.cat([counts, rates], 1)

    def entropy(self):
        relation, kernel, _, _ = self.weights()
        return mean_entropy(relation) + mean_entropy(kernel)


class GrammarModel(nn.Module):
    def __init__(self, n_count, n_rate, n_fixed, alphabet, hidden=128):
        super().__init__()
        self.grammar = Grammar(n_count, n_rate, alphabet)
        self.norm = nn.BatchNorm1d(n_count + n_rate)
        self.head = nn.Sequential(
            nn.Linear(n_count + n_rate + n_fixed, hidden), nn.ReLU(),
            nn.Linear(hidden, 64), nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, basis, fixed):
        return self.head(torch.cat([self.norm(self.grammar(basis)), fixed], 1)).squeeze(1)


class Inputs:
    def __init__(self, fixed_cols, alphabet):
        self.fixed_cols = fixed_cols
        self.alphabet = alphabet

    def fit(self, df):
        raw = matrix(df, self.alphabet.columns()).reshape(-1, *self.alphabet.shape)
        self.scale = (np.abs(raw).sum(axis=2, keepdims=True).mean(axis=0) + 1e-3).astype(np.float32)
        del raw
        x = matrix(df, self.fixed_cols)
        self.median = np.nan_to_num(np.nanmedian(x, axis=0))
        x = signed_log(np.where(np.isnan(x), self.median, x))
        self.mean, self.std = x.mean(0), x.std(0) + 1e-6
        return self

    def fixed(self, df):
        x = matrix(df, self.fixed_cols)
        x = signed_log(np.where(np.isnan(x), self.median, x))
        return torch.from_numpy(((x - self.mean) / self.std).astype(np.float32))

    def basis(self, df):
        x = matrix(df, self.alphabet.columns()).reshape(-1, *self.alphabet.shape)
        x /= self.scale
        return torch.from_numpy(x)


def batches(n, size, shuffle, generator=None):
    order = torch.randperm(n, generator=generator) if shuffle else torch.arange(n)
    for start in range(0, n, size):
        yield order[start:start + size]


@torch.no_grad()
def run_model(model, basis, fixed, device, size=32768):
    model.eval()
    logits, feats = [], []
    for idx in batches(len(basis), size, shuffle=False):
        b, f = basis[idx].to(device), fixed[idx].to(device)
        feats.append(model.grammar(b).cpu())
        logits.append(model(b, f).cpu())
    return torch.sigmoid(torch.cat(logits)).numpy(), torch.cat(feats).numpy()


def logloss(y, p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


def fit_grammar(train, val, fixed_cols, n_count, n_rate, device, alphabet, epochs=8, batch=8192, lr=3e-3, grammar_lr=5e-2,
                entropy_weight=1e-2, patience=2, seed=config.SEED, hard_from=None, hard_lr_factor=0.1):
    torch.manual_seed(seed)
    generator = torch.Generator().manual_seed(seed)
    inputs = Inputs(fixed_cols, alphabet).fit(train)
    tr_basis, tr_fixed = inputs.basis(train), inputs.fixed(train)
    tr_y = torch.from_numpy(train["label"].to_numpy().astype(np.float32))
    va_basis, va_fixed = inputs.basis(val), inputs.fixed(val)
    va_y = val["label"].to_numpy()

    model = GrammarModel(n_count, n_rate, len(fixed_cols), alphabet).to(device)
    grammar_params = list(model.grammar.parameters())
    other_params = [p for name, p in model.named_parameters() if not name.startswith("grammar.")]
    optimizer = torch.optim.AdamW([
        {"params": grammar_params, "lr": grammar_lr, "weight_decay": 0.0},
        {"params": other_params, "lr": lr, "weight_decay": 1e-5},
    ])
    loss_fn = nn.BCEWithLogitsLoss()
    best, best_state, bad, history = float("inf"), None, 0, []
    started = time.time()
    for epoch in range(epochs):
        if hard_from is not None and epoch == hard_from:
            model.grammar.hard = True
            optimizer.param_groups[0]["lr"] = grammar_lr * hard_lr_factor
            best, best_state, bad = float("inf"), None, 0
        model.train()
        for idx in batches(len(tr_y), batch, shuffle=True, generator=generator):
            b, f, y = tr_basis[idx].to(device), tr_fixed[idx].to(device), tr_y[idx].to(device)
            loss = loss_fn(model(b, f), y) + entropy_weight * model.grammar.entropy()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        val_pred, _ = run_model(model, va_basis, va_fixed, device)
        val_loss = logloss(va_y, val_pred)
        history.append({"epoch": epoch, "val_logloss": val_loss, "seconds": round(time.time() - started, 1)})
        print(f"grammar k={n_count + n_rate} epoch {epoch}: val logloss {val_loss:.4f}, {time.time() - started:.0f}s")
        if val_loss < best - 1e-5:
            best, bad = val_loss, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        elif hard_from is None or epoch >= hard_from:
            bad += 1
            if bad >= patience:
                break
    assert best_state is not None, "grammar training diverged"
    model.load_state_dict(best_state)
    return model, inputs, history


def save_grammar(path, model, inputs):
    torch.save({
        "state": model.state_dict(),
        "n_count": model.grammar.n_count,
        "hard": model.grammar.hard,
        "n_rate": model.grammar.relation.shape[0] - model.grammar.n_count,
        "alphabet": inputs.alphabet.name,
        "fixed_cols": inputs.fixed_cols,
        "scale": inputs.scale,
        "median": inputs.median,
        "mean": inputs.mean,
        "std": inputs.std,
    }, path)


def transform(model, inputs, df, device):
    return run_model(model, inputs.basis(df), inputs.fixed(df), device)


def describe(model, min_weight=0.1):
    relation, kernel, numerator, denominator = (w.detach().cpu().numpy() for w in model.grammar.weights())
    atoms, kernels = model.grammar.alphabet.atoms, model.grammar.alphabet.kernels
    rows = []
    for k in range(relation.shape[0]):
        kind = "count" if k < model.grammar.n_count else "rate"
        rel = sorted(((RELATIONS[i], float(w)) for i, w in enumerate(relation[k]) if w >= min_weight), key=lambda x: -x[1])
        ker = sorted(((kernels[i], float(w)) for i, w in enumerate(kernel[k]) if w >= min_weight), key=lambda x: -x[1])
        num = [atoms[i] for i, w in enumerate(numerator[k]) if w >= 0.5]
        den = [atoms[i] for i, w in enumerate(denominator[k]) if w >= 0.5]
        what = " + ".join(num) or "nothing"
        if kind == "rate":
            what = f"({what}) / ({' + '.join(den) or 'nothing'})"
        text = (
            f"{kind}: {what} over "
            + " + ".join(f"{w:.2f} {RELATION_TEXT[r]}" for r, w in rel)
            + ", kernel "
            + " + ".join(f"{w:.2f} {KERNEL_TEXT[m]}" for m, w in ker)
        )
        rows.append({
            "feature": k, "kind": kind, "text": text,
            "relation": dict(zip(RELATIONS, relation[k].round(4).tolist())),
            "kernel": dict(zip(kernels, kernel[k].round(4).tolist())),
            "numerator": dict(zip(atoms, numerator[k].round(4).tolist())),
            "denominator": dict(zip(atoms, denominator[k].round(4).tolist())),
        })
    return rows

In [ ]:
%%writefile /kaggle/working/code/rank/stats.py
import json
import re
from pathlib import Path

import numpy as np
import polars as pl
from scipy import stats

from .metrics import per_group_ndcg, per_user_auc

ALPHA = 0.05
POWER = 0.8
N_PERM = 100_000
PERM_CHUNK = 1000
TEST_METRICS = ("gauc", "ndcg@10", "logloss")
SEED_SUFFIX = re.compile(r"^(.*)_s(\d+)$")


def model_frame(df, col):
    return df.select(
        pl.col("uid").cast(pl.Int64),
        pl.col("days_before_end").alias("day"),
        pl.col("label").cast(pl.Int32),
        pl.col(col).cast(pl.Float64).clip(1e-6, 1 - 1e-6).alias("pred"),
    )


def user_terms(df, col):
    f = model_frame(df, col)
    auc = per_user_auc(f).select("uid", (pl.col("n") * pl.col("auc")).alias("value"), pl.col("n").cast(pl.Float64).alias("weight"))
    ndcg = per_group_ndcg(f).group_by("uid").agg(pl.col("ndcg").sum().alias("value"), pl.len().cast(pl.Float64).alias("weight"))
    loss = -(pl.col("label") * pl.col("pred").log() + (1 - pl.col("label")) * (1 - pl.col("pred")).log())
    logloss = f.group_by("uid").agg(loss.sum().alias("value"), pl.len().cast(pl.Float64).alias("weight"))
    return {m: t.sort("uid") for m, t in zip(TEST_METRICS, (auc, ndcg, logloss))}


def sign_flip_pvalue(s, delta, total_weight, n_perm=N_PERM, seed=0):
    rng = np.random.default_rng(seed)
    exceed, done = 0, 0
    while done < n_perm:
        k = min(PERM_CHUNK, n_perm - done)
        signs = rng.integers(0, 2, size=(k, len(s)), dtype=np.int8).astype(np.float64) * 2 - 1
        null = signs @ s / total_weight
        exceed += int((np.abs(null) >= abs(delta) * (1 - 1e-12)).sum())
        done += k
    return (exceed + 1) / (n_perm + 1)


def paired_test(s, w, n_perm=N_PERM, alpha=ALPHA, power=POWER):
    total = w.sum()
    delta = s.sum() / total
    n = len(s)
    se = np.sqrt(((s - w * delta) ** 2).sum() * n / (n - 1)) / total
    z = delta / se
    z_alpha, z_power = stats.norm.ppf(1 - alpha / 2), stats.norm.ppf(power)
    return {
        "delta": float(delta),
        "se": float(se),
        "ci_lo": float(delta - z_alpha * se),
        "ci_hi": float(delta + z_alpha * se),
        "z": float(z),
        "p_z": float(2 * stats.norm.sf(abs(z))),
        "p_perm": float(sign_flip_pvalue(s, delta, total, n_perm)),
        "mde": float((z_alpha + z_power) * se),
        "power_at_delta": float(stats.norm.sf(z_alpha - abs(z)) + stats.norm.cdf(-z_alpha - abs(z))),
        "n_users": int(n),
    }


def holm(pvalues):
    p = np.asarray(pvalues, dtype=np.float64)
    order = np.argsort(p)
    adjusted = np.empty_like(p)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (len(p) - rank) * p[i])
        adjusted[i] = min(running, 1.0)
    return adjusted


def comparisons_from_columns(columns, reference):
    runs = {}
    for col in columns:
        if not col.startswith("pred_"):
            continue
        name = col[len("pred_"):]
        match = SEED_SUFFIX.match(name)
        base, seed = (match.group(1), int(match.group(2))) if match else (name, None)
        runs.setdefault(base, {})[seed] = col
    ref = runs[reference]
    out = {}
    for base, by_seed in runs.items():
        if base == reference:
            continue
        common = sorted(set(by_seed) & set(ref), key=lambda x: -1 if x is None else x)
        if common:
            out[f"{base}_vs_{reference}"] = [(by_seed[s], ref[s]) for s in common]
    return out


def run_stat_tests(predictions_path, out_dir, reference="human", n_perm=N_PERM, alpha=ALPHA, power=POWER, models=None, name="stat_tests"):
    df = pl.read_parquet(predictions_path)
    comparisons = comparisons_from_columns(df.columns, reference)
    if models is not None:
        comparisons = {f"{m}_vs_{reference}": comparisons[f"{m}_vs_{reference}"] for m in models}
    cache = {}

    def terms(col):
        if col not in cache:
            cache[col] = user_terms(df, col)
        return cache[col]

    rows = []
    for comparison, pairs in comparisons.items():
        for metric in TEST_METRICS:
            diffs, per_seed, weight, uid = [], [], None, None
            for a, b in pairs:
                ta, tb = terms(a)[metric], terms(b)[metric]
                assert ta["uid"].equals(tb["uid"]), f"user sets differ for {a} and {b}"
                d = (ta["value"] - tb["value"]).to_numpy()
                weight = ta["weight"].to_numpy()
                uid = ta["uid"] if uid is None else uid
                assert uid.equals(ta["uid"]), "user sets differ between seeds"
                diffs.append(d)
                per_seed.append(float(d.sum() / weight.sum()))
            row = {"comparison": comparison, "metric": metric, "n_seeds": len(pairs),
                   **paired_test(np.mean(diffs, axis=0), weight, n_perm, alpha, power)}
            row["per_seed_delta"] = per_seed
            if len(pairs) > 1:
                t = stats.ttest_1samp(per_seed, 0.0)
                row["seed_t"], row["seed_p"] = float(t.statistic), float(t.pvalue)
            rows.append(row)
    adjusted = holm([r["p_perm"] for r in rows])
    for r, p in zip(rows, adjusted):
        r["p_holm"] = float(p)
        r["reject_h0"] = bool(p < alpha)
    out_dir = Path(out_dir)
    (out_dir / f"{name}.json").write_text(json.dumps(
        {"alpha": alpha, "power": power, "beta": 1 - power, "n_perm": n_perm, "reference": reference, "tests": rows}, indent=2))
    pl.DataFrame([{k: v for k, v in r.items() if k != "per_seed_delta"} for r in rows]).write_csv(out_dir / f"{name}.csv")
    return rows

In [ ]:
%%writefile /kaggle/working/code/rank/discovery.py
import gc
import json
import time
from pathlib import Path

import numpy as np
import polars as pl

from . import config
from .basis import ALPHABETS, RELATIONS, basis_name, degenerate_columns
from .experiments import results_table, slug
from .learned import describe, fit_grammar, save_grammar, transform
from .metrics import bootstrap, paired_delta, summarize
from .stats import run_stat_tests
from .train import fit_catboost, predict, refit_catboost

FIXED_PREFIXES = ("i_", "a_", "al_")
FIXED_EXTRA = [
    "c_hour", "c_weekday", "cf_user_score", "u_like_cnt_30d", "u_dislike_cnt_30d",
    "ui_like_state", "ui_dislike_state", "ua_liked_tracks", "ua_disliked_tracks",
]
REFERENCE = "human"
GRAMMAR_SIZES = (8, 16, 32)
BRUTE_SIZES = (8, 16, 32)
RANDOM_SIZE = 16
HYBRID_SIZE = 16
BRUTE_SAMPLE = 1_000_000
EVAL_COLS = ["uid", "days_before_end", "label"]


def fixed_and_human(meta):
    groups = meta["groups"]
    excluded = set(groups.get("decay", [])) | set(groups.get("basis", []))
    base = [c for c in meta["features"] if c not in excluded]
    fixed = [c for c in base if c.startswith(FIXED_PREFIXES) or c in FIXED_EXTRA]
    human = [c for c in base if c not in fixed]
    return fixed, human


def rate_candidates(alphabet):
    degenerate = degenerate_columns()
    out = {}
    for r in RELATIONS:
        for k in alphabet.kernels:
            if basis_name(r, "rec_lp", k) in degenerate:
                continue
            atom = {a: pl.col(basis_name(r, a, k)) for a in alphabet.atoms}
            listened = atom["rec_lp"] + atom["org_lp"]
            total = listened + atom["rec_skip"] + atom["org_skip"]
            recommended = atom["rec_lp"] + atom["rec_skip"]
            organic = atom["org_lp"] + atom["org_skip"]
            out[f"q_{r}_{k}_lp_rate"] = (listened + 0.5) / (total.clip(lower_bound=0) + 1)
            out[f"q_{r}_{k}_rec_lp_rate"] = (atom["rec_lp"] + 0.5) / (recommended.clip(lower_bound=0) + 1)
            out[f"q_{r}_{k}_org_share"] = (organic + 0.5) / (total.clip(lower_bound=0) + 1)
            if "ratio" in atom:
                out[f"q_{r}_{k}_mean_ratio"] = (atom["ratio"] + 0.5) / (total.clip(lower_bound=0) + 1)
            if "novel_lp" in atom:
                novel = atom["novel_lp"] + atom["novel_skip"]
                out[f"q_{r}_{k}_novel_lp_rate"] = (atom["novel_lp"] + 0.5) / (novel.clip(lower_bound=0) + 1)
    return out


def candidate_columns(alphabet):
    degenerate = degenerate_columns()
    return [c for c in alphabet.columns() if c not in degenerate] + list(rate_candidates(alphabet))


def view(df, cols, rates):
    return df.select([rates[c].cast(pl.Float32).alias(c) if c in rates else pl.col(c) for c in cols] + [pl.col("label")])


def importance_frame(model, cols):
    return pl.DataFrame({"feature": cols, "importance": model.get_feature_importance()}).sort("importance", descending=True)


def fit_set(name, cols, train, val, test, rates, task_type, iterations, out_dir, records, val_preds, test_preds, seed=config.SEED):
    tr, va = view(train, cols, rates), view(val, cols, rates)
    model, seconds = fit_catboost(tr, va, cols, task_type, iterations, seed)
    best = model.get_best_iteration() + 1
    val_preds[name] = predict(model, va, cols)
    importance = importance_frame(model, cols)
    del model
    trainval = pl.concat([tr, va], how="vertical")
    del tr, va
    final, final_seconds = refit_catboost(trainval, cols, task_type, best, seed)
    del trainval
    final.save_model(str(out_dir / "models" / f"{slug(name)}.cbm"))
    test_preds[name] = predict(final, view(test, cols, rates), cols)
    records[name] = {"features": len(cols), "best_iteration": best, "val_fit_s": round(seconds, 1), "final_fit_s": round(final_seconds, 1)}
    print(f"{name}: {len(cols)} features, best iteration {best}, fit {seconds:.0f}s + {final_seconds:.0f}s")
    gc.collect()
    return importance


def add_columns(df, names, values):
    return df.with_columns([pl.Series(n, values[:, j]) for j, n in enumerate(names)])


def save_predictions(out_dir, df, preds):
    cols = [pl.Series(f"pred_{name}", np.asarray(p, dtype=np.float32)) for name, p in preds.items()]
    df.select(EVAL_COLS + ["x_novelty"]).with_columns(cols).write_parquet(out_dir / "test_predictions.parquet")


def write_descriptions(out_dir, k, rows):
    (out_dir / "grammar" / f"features_k{k}.json").write_text(json.dumps(rows, indent=2))
    lines = [f"# Learned features, K = {k}", ""] + [f"{r['feature']:2d}. {r['text']}" for r in rows]
    (out_dir / "grammar" / f"features_k{k}.md").write_text("\n".join(lines) + "\n")


def fit_sampled(name, cols, train, val, test, rates, task_type, iterations, out_dir, records, val_preds, test_preds):
    sample = train.sample(n=min(BRUTE_SAMPLE, train.height), seed=config.SEED)
    va = view(val, cols, rates)
    model, seconds = fit_catboost(view(sample, cols, rates), va, cols, task_type, iterations)
    del sample
    val_preds[name] = predict(model, va, cols)
    del va
    test_preds[name] = predict(model, view(test, cols, rates), cols)
    model.save_model(str(out_dir / "models" / f"{slug(name)}.cbm"))
    records[name] = {"features": len(cols), "best_iteration": model.get_best_iteration() + 1, "val_fit_s": round(seconds, 1),
                     "train_rows": min(BRUTE_SAMPLE, train.height), "refit": False}
    print(f"{name}: {len(cols)} features on {records[name]['train_rows']} rows, best iteration {records[name]['best_iteration']}, fit {seconds:.0f}s")
    importance = importance_frame(model, cols)
    del model
    gc.collect()
    return importance


def load_frames(features_dir, alphabet):
    meta = json.loads((Path(features_dir) / "meta.json").read_text())
    assert meta.get("basis"), "features must be built with basis=True"
    fixed, human = fixed_and_human(meta)
    load_cols = list(dict.fromkeys(EVAL_COLS + fixed + human + alphabet.columns()))
    frames = [pl.read_parquet(Path(features_dir) / f"{s}.parquet", columns=load_cols) for s in config.SPLITS]
    for name, df in zip(config.SPLITS, frames):
        bad = [c for c in alphabet.columns() if not np.isfinite(df[c].to_numpy()).all()]
        assert not bad, f"{name}: non-finite basis columns {bad[:5]}"
    return fixed, human, frames


def make_dirs(out_dir):
    out_dir = Path(out_dir)
    for sub in ["models", "importance", "grammar"]:
        (out_dir / sub).mkdir(parents=True, exist_ok=True)
    return out_dir


def run_discovery(features_dir, out_dir, task_type="GPU", device="cuda", iterations=3000, n_boot=100, alphabet="rich"):
    out_dir = make_dirs(out_dir)
    alphabet = ALPHABETS[alphabet]
    fixed, human, (train, val, test) = load_frames(features_dir, alphabet)
    rates = rate_candidates(alphabet)
    candidates = candidate_columns(alphabet)
    print(f"alphabet {alphabet.name} {alphabet.shape}; train {train.height}, val {val.height}, test {test.height}; "
          f"fixed {len(fixed)}, human {len(human)}, candidates {len(candidates)}")

    records, val_preds, test_preds, timings = {}, {}, {}, {}
    run = dict(train=train, val=val, test=test, rates=rates, task_type=task_type, iterations=iterations, out_dir=out_dir,
               records=records, val_preds=val_preds, test_preds=test_preds)

    fit_set("fixed", fixed, **run)
    fit_set(REFERENCE, fixed + human, **run).write_csv(out_dir / "importance" / "human.csv")

    started = time.time()
    importance = fit_sampled("brute_all", fixed + candidates, **run)
    timings["brute_search_s"] = round(time.time() - started, 1)
    importance.write_csv(out_dir / "importance" / "brute_all.csv")
    ranked = importance.filter(pl.col("feature").is_in(candidates))["feature"].to_list()
    for k in BRUTE_SIZES:
        fit_set(f"brute_top{k}", fixed + ranked[:k], **run)
    random_pick = list(np.random.default_rng(config.SEED).choice(candidates, RANDOM_SIZE, replace=False))
    fit_set(f"random{RANDOM_SIZE}", fixed + random_pick, **run)

    grammar_history = {}
    for k in GRAMMAR_SIZES:
        started = time.time()
        model, inputs, history = fit_grammar(train, val, fixed, k // 2, k - k // 2, device, alphabet)
        timings[f"grammar{k}_fit_s"] = round(time.time() - started, 1)
        grammar_history[k] = history
        names = [f"g{k}_{j:02d}" for j in range(k)]
        train = add_columns(train, names, transform(model, inputs, train, device)[1])
        val_preds[f"grammar_nn{k}"], val_feats = transform(model, inputs, val, device)
        val = add_columns(val, names, val_feats)
        test_preds[f"grammar_nn{k}"], test_feats = transform(model, inputs, test, device)
        test = add_columns(test, names, test_feats)
        write_descriptions(out_dir, k, describe(model))
        save_grammar(out_dir / "grammar" / f"model_k{k}.pt", model, inputs)
        run.update(train=train, val=val, test=test)
        fit_set(f"grammar{k}", fixed + names, **run).write_csv(out_dir / "importance" / f"grammar{k}.csv")
        del model, inputs
        gc.collect()

    hybrid = f"human+grammar{HYBRID_SIZE}"
    fit_set(hybrid, fixed + human + [f"g{HYBRID_SIZE}_{j:02d}" for j in range(HYBRID_SIZE)], **run).write_csv(
        out_dir / "importance" / f"{slug(hybrid)}.csv"
    )

    val_table = results_table(val_preds, val)
    val_table.write_csv(out_dir / "val_metrics.csv")
    test_table = results_table(test_preds, test)
    test_table.write_csv(out_dir / "test_metrics.csv")
    save_predictions(out_dir, test, test_preds)
    print(test_table.filter(pl.col("slice") == "all").select("model", "logloss", "auc", "gauc", "ndcg@10"))

    boot_models = [REFERENCE, "fixed", "brute_all", f"brute_top{HYBRID_SIZE}", f"random{RANDOM_SIZE}",
                   f"grammar{HYBRID_SIZE}", f"grammar_nn{HYBRID_SIZE}", hybrid] + [f"grammar{k}" for k in GRAMMAR_SIZES if k != HYBRID_SIZE]
    samples = bootstrap(test, {k: test_preds[k] for k in boot_models}, n_boot=n_boot)
    summary = {
        "reference": REFERENCE,
        "alphabet": {"name": alphabet.name, "atoms": alphabet.atoms, "kernels": alphabet.kernels, "candidates": len(candidates)},
        "fixed": fixed,
        "human": human,
        "brute_ranking_top32": ranked[:32],
        "random_pick": random_pick,
        "records": records,
        "timings": timings,
        "grammar_history": grammar_history,
        "test_bootstrap_ci": summarize(samples),
        "test_paired_deltas": {f"{m}_vs_{REFERENCE}": paired_delta(samples, m, REFERENCE) for m in boot_models if m != REFERENCE},
        "n_boot": n_boot,
    }
    summary["stat_tests"] = run_stat_tests(out_dir / "test_predictions.parquet", out_dir, reference=REFERENCE)
    (out_dir / "summary.json").write_text(json.dumps(summary, indent=2))
    return summary

In [ ]:
%%writefile /kaggle/working/code/rank/seeds.py
import gc
import json
import time
from itertools import combinations

import numpy as np
import polars as pl

from . import config
from .basis import ALPHABETS
from .discovery import REFERENCE, add_columns, fit_set, load_frames, make_dirs, save_predictions, write_descriptions
from .experiments import results_table
from .learned import describe, fit_grammar, save_grammar, transform
from .metrics import METRICS, bootstrap, paired_delta, summarize
from .stats import run_stat_tests

SEEDS = (42, 7, 13, 101, 2024)
GRAMMAR_K = 32
HYBRID_K = 16
STABILITY_ROWS = 100_000


def seed_name(model, seed):
    return f"{model}_s{seed}"


def mean_over_seeds(samples, model, seeds):
    runs = [samples[seed_name(model, s)] for s in seeds]
    return [{m: float(np.mean([r[i][m] for r in runs])) for m in METRICS} for i in range(len(runs[0]))]


def value_overlap(a, b):
    k = a.shape[1]
    corr = np.nan_to_num(np.corrcoef(a.T, b.T)[:k, k:])
    return float(np.abs(corr).max(axis=1).mean())


def signature(rows):
    return {(max(r["relation"], key=r["relation"].get), max(r["kernel"], key=r["kernel"].get), r["kind"]) for r in rows}


def stability(val_feats, descriptions):
    rows = []
    for a, b in combinations(sorted(val_feats), 2):
        sa, sb = signature(descriptions[a]), signature(descriptions[b])
        rows.append({
            "seed_a": a, "seed_b": b,
            "value_overlap": value_overlap(val_feats[a], val_feats[b]),
            "signature_jaccard": len(sa & sb) / len(sa | sb),
        })
    return rows


def run_seeds(features_dir, out_dir, task_type="GPU", device="cuda", iterations=3000, n_boot=100, alphabet="rich", seeds=SEEDS):
    out_dir = make_dirs(out_dir)
    alphabet = ALPHABETS[alphabet]
    fixed, human, (train, val, test) = load_frames(features_dir, alphabet)
    rows = np.sort(np.random.default_rng(config.SEED).choice(val.height, min(STABILITY_ROWS, val.height), replace=False))
    print(f"alphabet {alphabet.name}; train {train.height}, val {val.height}, test {test.height}; seeds {list(seeds)}")

    records, val_preds, test_preds, timings = {}, {}, {}, {}
    val_feats, descriptions = {}, {}
    hybrid = f"human+grammar{HYBRID_K}"
    for seed in seeds:
        started = time.time()
        run = dict(train=train, val=val, test=test, rates={}, task_type=task_type, iterations=iterations, out_dir=out_dir,
                   records=records, val_preds=val_preds, test_preds=test_preds, seed=seed)
        fit_set(seed_name(REFERENCE, seed), fixed + human, **run)
        names = {}
        for k in (HYBRID_K, GRAMMAR_K):
            model, inputs, _ = fit_grammar(train, val, fixed, k // 2, k - k // 2, device, alphabet, seed=seed)
            names[k] = [f"g{k}_s{seed}_{j:02d}" for j in range(k)]
            train = add_columns(train, names[k], transform(model, inputs, train, device)[1])
            val_preds[seed_name(f"grammar_nn{k}", seed)], feats = transform(model, inputs, val, device)
            val = add_columns(val, names[k], feats)
            test_preds[seed_name(f"grammar_nn{k}", seed)], feats = transform(model, inputs, test, device)
            test = add_columns(test, names[k], feats)
            rows_k = describe(model)
            write_descriptions(out_dir, f"{k}_s{seed}", rows_k)
            save_grammar(out_dir / "grammar" / f"model_k{k}_s{seed}.pt", model, inputs)
            if k == GRAMMAR_K:
                val_feats[seed] = val.select(names[k]).to_numpy()[rows]
                descriptions[seed] = rows_k
            del model, inputs
            gc.collect()
        run.update(train=train, val=val, test=test)
        fit_set(seed_name(f"grammar{GRAMMAR_K}", seed), fixed + names[GRAMMAR_K], **run)
        fit_set(seed_name(hybrid, seed), fixed + human + names[HYBRID_K], **run)
        added = names[HYBRID_K] + names[GRAMMAR_K]
        train, val, test = (df.drop(added) for df in (train, val, test))
        gc.collect()
        timings[f"seed_{seed}_s"] = round(time.time() - started, 1)
        print(f"seed {seed} done in {timings[f'seed_{seed}_s']:.0f}s")

    val_table = results_table(val_preds, val)
    val_table.write_csv(out_dir / "val_metrics.csv")
    test_table = results_table(test_preds, test)
    test_table.write_csv(out_dir / "test_metrics.csv")
    save_predictions(out_dir, test, test_preds)

    samples = bootstrap(test, test_preds, n_boot=n_boot)
    compared = [f"grammar{GRAMMAR_K}", hybrid, f"grammar_nn{GRAMMAR_K}"]
    per_seed = {
        f"{m}_vs_{REFERENCE}": {str(s): paired_delta(samples, seed_name(m, s), seed_name(REFERENCE, s)) for s in seeds}
        for m in compared
    }
    for m in compared + [REFERENCE]:
        samples[f"{m}_mean"] = mean_over_seeds(samples, m, seeds)
    seed_mean = {f"{m}_vs_{REFERENCE}": paired_delta(samples, f"{m}_mean", f"{REFERENCE}_mean") for m in compared}

    point = test_table.filter(pl.col("slice") == "all")
    spread = {}
    for m in compared + [REFERENCE]:
        for metric in METRICS:
            values = [point.filter(pl.col("model") == seed_name(m, s))[metric].item() for s in seeds]
            spread.setdefault(m, {})[metric] = {"mean": float(np.mean(values)), "std": float(np.std(values, ddof=1)), "values": values}
    wins = {
        m: sum(per_seed[f"{m}_vs_{REFERENCE}"][str(s)]["gauc"]["mean"] > 0 for s in seeds) for m in compared
    }

    stab = stability(val_feats, descriptions)
    pl.DataFrame(stab).write_csv(out_dir / "stability.csv")
    summary = {
        "reference": REFERENCE,
        "alphabet": alphabet.name,
        "seeds": list(seeds),
        "records": records,
        "timings": timings,
        "test_point_spread": spread,
        "test_seed_mean_deltas": seed_mean,
        "test_per_seed_deltas": per_seed,
        "gauc_wins_vs_reference": wins,
        "stability_mean": {k: float(np.mean([r[k] for r in stab])) for k in ["value_overlap", "signature_jaccard"]},
        "test_bootstrap_ci": summarize({k: v for k, v in samples.items() if k.endswith("_mean")}),
        "n_boot": n_boot,
    }
    summary["stat_tests"] = run_stat_tests(out_dir / "test_predictions.parquet", out_dir, reference=REFERENCE, models=compared)
    (out_dir / "summary.json").write_text(json.dumps(summary, indent=2))
    return summary

In [ ]:
%%writefile /kaggle/working/code/rank/alphabet.py
from itertools import combinations

from .basis import ALPHABETS, ATOMS, KERNELS, RECENCY_KERNELS, Alphabet

CHANNELS = {
    "ratio": (["ratio"], []),
    "novel": (["novel_lp", "novel_skip"], []),
    "sess": ([], ["sess"]),
    "recency": ([], list(RECENCY_KERNELS)),
}


def alphabet_with(channels, name):
    drop_atoms = {a for c in CHANNELS if c not in channels for a in CHANNELS[c][0]}
    drop_kernels = {k for c in CHANNELS if c not in channels for k in CHANNELS[c][1]}
    return Alphabet(name, [a for a in ATOMS if a not in drop_atoms], [k for k in KERNELS if k not in drop_kernels])


def ablation_alphabets():
    out = {"rich": alphabet_with(tuple(CHANNELS), "rich"), "base": alphabet_with((), "base")}
    for c in CHANNELS:
        out[f"rich-{c}"] = alphabet_with(tuple(x for x in CHANNELS if x != c), f"rich-{c}")
        out[f"base+{c}"] = alphabet_with((c,), f"base+{c}")
    for name in ["rich", "base"]:
        assert out[name].columns() == ALPHABETS[name].columns()
    return out

In [ ]:
%%writefile /kaggle/working/code/rank/ablation.py
import gc
import json
import time

import numpy as np
import polars as pl

from .alphabet import CHANNELS, ablation_alphabets
from .basis import ALPHABETS
from .discovery import REFERENCE, add_columns, fit_set, load_frames, make_dirs, save_predictions, write_descriptions
from .experiments import results_table
from .learned import describe, fit_grammar, save_grammar, transform
from .metrics import METRICS
from .stats import run_stat_tests

SEEDS = (42, 7)
K = 32


def model_name(variant):
    return f"g{K}_{variant}"


def channel_effects(point, seeds):
    def value(variant, metric):
        return float(np.mean([point[f"{model_name(variant)}_s{s}"][metric] for s in seeds]))

    rows = []
    for c in CHANNELS:
        for metric in METRICS:
            rows.append({
                "channel": c, "metric": metric,
                "drop_from_rich": value(f"rich-{c}", metric) - value("rich", metric),
                "add_to_base": value(f"base+{c}", metric) - value("base", metric),
            })
    return rows


def run_ablation(features_dir, out_dir, task_type="GPU", device="cuda", iterations=3000, seeds=SEEDS):
    out_dir = make_dirs(out_dir)
    alphabets = ablation_alphabets()
    fixed, human, (train, val, test) = load_frames(features_dir, ALPHABETS["rich"])
    print(f"train {train.height}, val {val.height}, test {test.height}; variants {list(alphabets)}; seeds {list(seeds)}")

    records, val_preds, test_preds, timings = {}, {}, {}, {}
    for seed in seeds:
        run = dict(train=train, val=val, test=test, rates={}, task_type=task_type, iterations=iterations, out_dir=out_dir,
                   records=records, val_preds=val_preds, test_preds=test_preds, seed=seed)
        fit_set(f"{REFERENCE}_s{seed}", fixed + human, **run)
        for variant, alphabet in alphabets.items():
            started = time.time()
            model, inputs, _ = fit_grammar(train, val, fixed, K // 2, K - K // 2, device, alphabet, seed=seed)
            name = f"{model_name(variant)}_s{seed}"
            cols = [f"{name}_{j:02d}" for j in range(K)]
            train = add_columns(train, cols, transform(model, inputs, train, device)[1])
            val = add_columns(val, cols, transform(model, inputs, val, device)[1])
            test = add_columns(test, cols, transform(model, inputs, test, device)[1])
            write_descriptions(out_dir, f"{K}_{variant}_s{seed}", describe(model))
            save_grammar(out_dir / "grammar" / f"model_k{K}_{variant}_s{seed}.pt", model, inputs)
            del model, inputs
            run.update(train=train, val=val, test=test)
            fit_set(name, fixed + cols, **run)
            train, val, test = (df.drop(cols) for df in (train, val, test))
            run.update(train=train, val=val, test=test)
            gc.collect()
            timings[name] = round(time.time() - started, 1)
            print(f"{name} done in {timings[name]:.0f}s")

    val_table = results_table(val_preds, val)
    val_table.write_csv(out_dir / "val_metrics.csv")
    test_table = results_table(test_preds, test)
    test_table.write_csv(out_dir / "test_metrics.csv")
    save_predictions(out_dir, test, test_preds)

    point = {r["model"]: r for r in test_table.filter(pl.col("slice") == "all").to_dicts()}
    effects = channel_effects(point, seeds)
    pl.DataFrame(effects).write_csv(out_dir / "channel_effects.csv")
    variants = [model_name(v) for v in alphabets if v != "rich"]
    vs_rich = run_stat_tests(out_dir / "test_predictions.parquet", out_dir, reference=model_name("rich"), models=variants,
                             name="stat_tests_vs_rich")
    vs_human = run_stat_tests(out_dir / "test_predictions.parquet", out_dir, reference=REFERENCE,
                              models=[model_name(v) for v in alphabets], name="stat_tests_vs_human")
    summary = {
        "reference": REFERENCE,
        "seeds": list(seeds),
        "k": K,
        "alphabets": {k: {"atoms": a.atoms, "kernels": a.kernels} for k, a in alphabets.items()},
        "records": records,
        "timings": timings,
        "channel_effects": effects,
        "stat_tests_vs_rich": vs_rich,
        "stat_tests_vs_human": vs_human,
    }
    (out_dir / "summary.json").write_text(json.dumps(summary, indent=2))
    return summary

In [ ]:
%%writefile /kaggle/working/code/rank/prune.py
import gc
import json
from pathlib import Path

import numpy as np
import polars as pl
import torch

from .basis import ALPHABETS, RELATIONS
from .discovery import REFERENCE, add_columns, fit_set, load_frames, make_dirs, save_predictions
from .experiments import results_table
from .learned import GrammarModel, Inputs
from .stats import run_stat_tests

SEEDS = (42, 7)
K = 32
THRESHOLDS = (0.001, 0.01, 0.05)
VARIANTS = ["dense"] + [f"t{t}" for t in THRESHOLDS] + ["readable"]
BATCH = 65536


def load_grammar(path):
    ckpt = torch.load(path, weights_only=False, map_location="cpu")
    alphabet = ALPHABETS[ckpt["alphabet"]]
    model = GrammarModel(ckpt["n_count"], ckpt["n_rate"], len(ckpt["fixed_cols"]), alphabet)
    model.load_state_dict(ckpt["state"])
    model.grammar.hard = ckpt.get("hard", False)
    model.eval()
    inputs = Inputs(ckpt["fixed_cols"], alphabet)
    for key in ["scale", "median", "mean", "std"]:
        setattr(inputs, key, ckpt[key])
    return model, inputs


@torch.no_grad()
def cell_weights(model):
    relation, kernel, numerator, denominator = (w.cpu() for w in model.grammar.weights())
    top = torch.einsum("fr,fm,fe->frem", relation, kernel, numerator)
    bottom = torch.einsum("fr,fm,fe->frem", relation, kernel, denominator)
    bottom[:model.grammar.n_count] = 0
    return top, bottom


def threshold_weights(model, threshold):
    top, bottom = cell_weights(model)
    return torch.where(top >= threshold, top, 0), torch.where(bottom >= threshold, bottom, 0)


@torch.no_grad()
def readable_weights(model):
    relation, kernel, numerator, denominator = (w.cpu() for w in model.grammar.weights())
    top = torch.zeros(relation.shape[0], *model.grammar.alphabet.shape)
    bottom = torch.zeros_like(top)
    r, m = relation.argmax(1), kernel.argmax(1)
    for f in range(relation.shape[0]):
        top[f, r[f], :, m[f]] = (numerator[f] >= 0.5).float()
        if f >= model.grammar.n_count:
            bottom[f, r[f], :, m[f]] = (denominator[f] >= 0.5).float()
    return top, bottom


def variant_weights(model):
    out = {"dense": threshold_weights(model, 0.0)}
    for t in THRESHOLDS:
        out[f"t{t}"] = threshold_weights(model, t)
    out["readable"] = readable_weights(model)
    return out


@torch.no_grad()
def features_from_weights(basis, top, bottom, n_count, device="cpu"):
    top, bottom = top.to(device), bottom.to(device)
    out = []
    for start in range(0, len(basis), BATCH):
        b = basis[start:start + BATCH].to(device)
        num = torch.einsum("brem,frem->bf", b, top)
        den = torch.einsum("brem,frem->bf", b, bottom)
        counts = torch.log1p(num[:, :n_count].clamp(min=0))
        rates = (num[:, n_count:] + 0.5) / (den[:, n_count:].clamp(min=0) + 1)
        out.append(torch.cat([counts, rates], 1).cpu())
    return torch.cat(out).numpy()


def used_cells(top, bottom, alphabet):
    used = ((top > 0) | (bottom > 0)).any(0).numpy()
    return [(RELATIONS[r], alphabet.atoms[e], alphabet.kernels[m]) for r, e, m in zip(*np.nonzero(used))]


def find_weights(root, seed):
    return next(Path(root).rglob(f"model_k{K}_s{seed}.pt"))


def run_prune(features_dir, weights_root, out_dir, task_type="GPU", device="cuda", iterations=3000, seeds=SEEDS, variants=tuple(VARIANTS)):
    out_dir = make_dirs(out_dir)
    alphabet = ALPHABETS["rich"]
    fixed, human, (train, val, test) = load_frames(features_dir, alphabet)
    records, val_preds, test_preds, cells = {}, {}, {}, {}
    for seed in seeds:
        run = dict(train=train, val=val, test=test, rates={}, task_type=task_type, iterations=iterations, out_dir=out_dir,
                   records=records, val_preds=val_preds, test_preds=test_preds, seed=seed)
        fit_set(f"{REFERENCE}_s{seed}", fixed + human, **run)
        model, inputs = load_grammar(find_weights(weights_root, seed))
        assert inputs.fixed_cols == fixed, "fixed columns differ from the grammar checkpoint"
        basis = {name: inputs.basis(df) for name, df in zip(["train", "val", "test"], (train, val, test))}
        for variant, (top, bottom) in variant_weights(model).items():
            if variant not in variants:
                continue
            cells[f"{variant}_s{seed}"] = used_cells(top, bottom, alphabet)
            name = f"g{K}_{variant}_s{seed}"
            cols = [f"{name}_{j:02d}" for j in range(K)]
            feats = {s: features_from_weights(basis[s], top, bottom, model.grammar.n_count, device) for s in basis}
            train, val, test = (add_columns(df, cols, feats[s]) for s, df in zip(["train", "val", "test"], (train, val, test)))
            run.update(train=train, val=val, test=test)
            fit_set(name, fixed + cols, **run)
            train, val, test = (df.drop(cols) for df in (train, val, test))
            run.update(train=train, val=val, test=test)
            print(f"{name}: {len(cells[f'{variant}_s{seed}'])} cells")
            del feats
            gc.collect()
        del basis, model, inputs
        gc.collect()

    val_table = results_table(val_preds, val)
    val_table.write_csv(out_dir / "val_metrics.csv")
    test_table = results_table(test_preds, test)
    test_table.write_csv(out_dir / "test_metrics.csv")
    save_predictions(out_dir, test, test_preds)
    vs_dense = run_stat_tests(out_dir / "test_predictions.parquet", out_dir, reference=f"g{K}_dense",
                              models=[f"g{K}_{v}" for v in variants if v != "dense"], name="stat_tests_vs_dense")
    vs_human = run_stat_tests(out_dir / "test_predictions.parquet", out_dir, reference=REFERENCE,
                              models=[f"g{K}_{v}" for v in variants], name="stat_tests_vs_human")
    summary = {
        "reference": REFERENCE,
        "seeds": list(seeds),
        "thresholds": list(THRESHOLDS),
        "records": records,
        "cells": cells,
        "cells_count": {k: len(v) for k, v in cells.items()},
        "stat_tests_vs_dense": vs_dense,
        "stat_tests_vs_human": vs_human,
    }
    (out_dir / "summary.json").write_text(json.dumps(summary, indent=2))
    return summary

In [ ]:
%%writefile /kaggle/working/code/rank/sparse.py
import gc
import json

import polars as pl

from .basis import ALPHABETS
from .discovery import REFERENCE, add_columns, fit_set, load_frames, make_dirs, save_predictions, write_descriptions
from .experiments import results_table
from .learned import describe, fit_grammar, save_grammar, transform
from .prune import threshold_weights, used_cells
from .stats import run_stat_tests

SEEDS = (42, 7, 13)
K = 32
MODES = {
    "dense": dict(),
    "hard": dict(epochs=10, hard_from=4, hard_lr_factor=0.1),
    "frozen": dict(epochs=8, hard_from=4, hard_lr_factor=0.0),
}


def run_sparse(features_dir, out_dir, task_type="GPU", device="cuda", iterations=3000, seeds=SEEDS):
    out_dir = make_dirs(out_dir)
    alphabet = ALPHABETS["rich"]
    fixed, human, (train, val, test) = load_frames(features_dir, alphabet)
    records, val_preds, test_preds, cells, history = {}, {}, {}, {}, {}
    for seed in seeds:
        run = dict(train=train, val=val, test=test, rates={}, task_type=task_type, iterations=iterations, out_dir=out_dir,
                   records=records, val_preds=val_preds, test_preds=test_preds, seed=seed)
        fit_set(f"{REFERENCE}_s{seed}", fixed + human, **run)
        for mode, kwargs in MODES.items():
            model, inputs, history[f"{mode}_s{seed}"] = fit_grammar(train, val, fixed, K // 2, K - K // 2, device, alphabet, seed=seed, **kwargs)
            name = f"g{K}_{mode}_s{seed}"
            cells[name] = used_cells(*threshold_weights(model, 1e-12), alphabet)
            write_descriptions(out_dir, f"{K}_{mode}_s{seed}", describe(model))
            save_grammar(out_dir / "grammar" / f"model_k{K}_{mode}_s{seed}.pt", model, inputs)
            cols = [f"{name}_{j:02d}" for j in range(K)]
            train, val, test = (add_columns(df, cols, transform(model, inputs, df, device)[1]) for df in (train, val, test))
            del model, inputs
            run.update(train=train, val=val, test=test)
            fit_set(name, fixed + cols, **run)
            train, val, test = (df.drop(cols) for df in (train, val, test))
            run.update(train=train, val=val, test=test)
            print(f"{name}: {len(cells[name])} cells")
            gc.collect()

    results_table(val_preds, val).write_csv(out_dir / "val_metrics.csv")
    results_table(test_preds, test).write_csv(out_dir / "test_metrics.csv")
    save_predictions(out_dir, test, test_preds)
    vs_dense = run_stat_tests(out_dir / "test_predictions.parquet", out_dir, reference=f"g{K}_dense", models=[f"g{K}_{m}" for m in MODES if m != "dense"],
                              name="stat_tests_vs_dense")
    vs_human = run_stat_tests(out_dir / "test_predictions.parquet", out_dir, reference=REFERENCE,
                              models=[f"g{K}_{m}" for m in MODES], name="stat_tests_vs_human")
    summary = {
        "reference": REFERENCE, "seeds": list(seeds), "modes": MODES, "records": records,
        "cells": cells, "cells_count": {k: len(v) for k, v in cells.items()}, "grammar_history": history,
        "stat_tests_vs_dense": vs_dense, "stat_tests_vs_human": vs_human,
    }
    (out_dir / "summary.json").write_text(json.dumps(summary, indent=2))
    return summary

In [ ]:
%%writefile /kaggle/working/code/rank/scale.py
import gc
import json

from .basis import ALPHABETS
from .discovery import REFERENCE, add_columns, fit_set, load_frames, make_dirs, save_predictions, write_descriptions
from .experiments import results_table
from .learned import describe, fit_grammar, save_grammar, transform
from .stats import run_stat_tests

SEEDS = (42, 7, 13)
SIZES = (32, 64)


def run_scale(features_dir, out_dir, task_type="GPU", device="cuda", iterations=3000, seeds=SEEDS, sizes=SIZES):
    out_dir = make_dirs(out_dir)
    alphabet = ALPHABETS["rich"]
    fixed, human, (train, val, test) = load_frames(features_dir, alphabet)
    records, val_preds, test_preds, history = {}, {}, {}, {}
    for seed in seeds:
        run = dict(train=train, val=val, test=test, rates={}, task_type=task_type, iterations=iterations, out_dir=out_dir,
                   records=records, val_preds=val_preds, test_preds=test_preds, seed=seed)
        fit_set(f"{REFERENCE}_s{seed}", fixed + human, **run)
        added = []
        for k in sizes:
            model, inputs, history[f"k{k}_s{seed}"] = fit_grammar(train, val, fixed, k // 2, k - k // 2, device, alphabet, seed=seed)
            cols = [f"g{k}_s{seed}_{j:02d}" for j in range(k)]
            write_descriptions(out_dir, f"{k}_s{seed}", describe(model))
            save_grammar(out_dir / "grammar" / f"model_k{k}_s{seed}.pt", model, inputs)
            train, val, test = (add_columns(df, cols, transform(model, inputs, df, device)[1]) for df in (train, val, test))
            del model, inputs
            added += cols
            run.update(train=train, val=val, test=test)
            fit_set(f"grammar{k}_s{seed}", fixed + cols, **run)
            fit_set(f"human+grammar{k}_s{seed}", fixed + human + cols, **run)
            gc.collect()
        train, val, test = (df.drop(added) for df in (train, val, test))
        gc.collect()

    results_table(val_preds, val).write_csv(out_dir / "val_metrics.csv")
    results_table(test_preds, test).write_csv(out_dir / "test_metrics.csv")
    save_predictions(out_dir, test, test_preds)
    models = [m for k in sizes for m in (f"grammar{k}", f"human+grammar{k}")]
    vs_human = run_stat_tests(out_dir / "test_predictions.parquet", out_dir, reference=REFERENCE, models=models,
                              name="stat_tests_vs_human")
    big = max(sizes)
    vs_small = run_stat_tests(out_dir / "test_predictions.parquet", out_dir, reference=f"grammar{min(sizes)}",
                              models=[f"grammar{big}"], name="stat_tests_k")
    vs_small_hybrid = run_stat_tests(out_dir / "test_predictions.parquet", out_dir, reference=f"human+grammar{min(sizes)}",
                                     models=[f"human+grammar{big}"], name="stat_tests_hybrid_k")
    summary = {
        "reference": REFERENCE, "seeds": list(seeds), "sizes": list(sizes), "records": records, "grammar_history": history,
        "stat_tests_vs_human": vs_human, "stat_tests_k": vs_small, "stat_tests_hybrid_k": vs_small_hybrid,
    }
    (out_dir / "summary.json").write_text(json.dumps(summary, indent=2))
    return summary

In [ ]:
import catboost
import polars as pl
import torch
from catboost.utils import get_gpu_device_count

from rank.features import build_features, code_hash, dataset_end, find_cached, history_prior
from rank.leakage import check_leakage
from rank.prepare import prepare_listens

TASK_TYPE = "GPU" if get_gpu_device_count() > 0 else "CPU"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FEATURE_KWARGS = dict(taus=(), basis=True)
STAGE = "scale"
FEATURES_OUT = Path("/kaggle/working/features") if STAGE == "features" else WORK / "features"
FEATURES = None if STAGE == "features" else find_cached("/kaggle/input", **FEATURE_KWARGS)
print(f"feature code hash {code_hash()}, cached features: {FEATURES}")
pl.Config.set_tbl_rows(60)
pl.Config.set_tbl_cols(12)
pl.Config.set_tbl_width_chars(200)
print(f"polars {pl.__version__}, catboost {catboost.__version__}, torch {torch.__version__}, task type {TASK_TYPE}, device {DEVICE}")
timings = {}

## 1. Подготовка прослушиваний

In [ ]:
if FEATURES is None:
    started = time.time()
    prepare_listens(events_path, mappings_dir, WORK / "listens")
    timings["prepare_s"] = round(time.time() - started, 1)
    t_end = dataset_end(events_path)
    prior = history_prior(pl.scan_parquet(str(WORK / "listens" / "part_*.parquet")), t_end)
    print(f"t_end {t_end}, prior {prior:.4f}")
else:
    print(f"using cached features from {FEATURES}")

## 2. Проверка утечек

Для первой тысячи пользователей признаки считаются по полным данным и по данным, обрезанным в середине валидации; для объектов за сутки до обрезки все признаки должны совпасть.
Канарейка `canary_next_ratio` смотрит в будущее, и тест обязан её поймать.

In [ ]:
if FEATURES is None:
    started = time.time()
    leak = check_leakage(WORK / "listens", events_path, mappings_dir, WORK / "leakage", t_end, prior, max_parts=1, **FEATURE_KWARGS)
    timings["leakage_check_s"] = round(time.time() - started, 1)
else:
    leak = json.loads((FEATURES / "leakage_report.json").read_text())
(OUT / "leakage_report.json").write_text(json.dumps(leak, indent=2))
print(json.dumps({k: v for k, v in leak.items() if k != "single_feature_auc_top10"}, indent=2))
assert leak["passed"], "leakage check failed"

## 3. Признаки

In [ ]:
if FEATURES is None:
    started = time.time()
    meta = build_features(WORK / "listens", events_path, mappings_dir, FEATURES_OUT, t_end=t_end, prior=prior, **FEATURE_KWARGS)
    timings["features_s"] = round(time.time() - started, 1)
    (FEATURES_OUT / "leakage_report.json").write_text(json.dumps(leak, indent=2))
    FEATURES = FEATURES_OUT
else:
    meta = json.loads((FEATURES / "meta.json").read_text())
(OUT / "features_meta.json").write_text(json.dumps(meta, indent=2))
print({g: len(c) for g, c in meta["groups"].items()})

## 4. Эксперименты

In [ ]:
from rank.scale import run_scale

started = time.time()
summary = run_scale(FEATURES, OUT, task_type=TASK_TYPE, device=DEVICE, iterations=3000)
timings["train_eval_s"] = round(time.time() - started, 1)

In [ ]:
cols = ["model", "logloss", "auc", "gauc", "ndcg@10"]
print(pl.read_csv(OUT / "test_metrics.csv").filter(pl.col("slice") == "all").select(cols).with_columns(pl.col(pl.Float64).round(4)))
for name in ["stat_tests_vs_human", "stat_tests_k", "stat_tests_hybrid_k"]:
    print(name)
    print(pl.read_csv(OUT / f"{name}.csv").select(
        "comparison", "metric", "delta", "ci_lo", "ci_hi", "p_perm", "p_holm", "mde", "reject_h0"
    ).with_columns(pl.col(pl.Float64).round(5)))

In [ ]:
run_info = {
    "timings": timings,
    "feature_code_hash": code_hash(),
    "cached_features": str(FEATURES) if STAGE != "features" else None,
    "records": globals().get("summary", {}).get("records", {}),
    "task_type": TASK_TYPE,
    "polars": pl.__version__,
    "catboost": catboost.__version__,
    "torch": torch.__version__,
    "gpu": GPU_INFO,
}
(OUT / "run_info.json").write_text(json.dumps(run_info, indent=2))
print(json.dumps(run_info, indent=2))